# <font face="BIZ UDゴシック">プリプロセッサ・属性・名前空間・キャスト</font>



## キーポイント

* プリプロセッサはコンパイラの実行の前に、`#include`や`#define`などの「プリプロセッサ指令」を処理する
* マクロを使うとプログラム中のテキストを自由に置換できる
* マクロは次のように書く
  ```cpp
  #define AAA 123       // マクロ
  #define BBB(s, t) t s // 関数型マクロ
  ```
* `[[fallthrough]]`属性を使うと、`switch`文の`case`ラベルに`break`文を書かなくても警告されなくなる
* 「名前空間」を使うと、関数名やクラス名の重複を防げる
*  名前空間を省略したい場合は`using`ディレクティブを使う
* C++形式のキャストには以下の4種類がある<br>
  * `static_cast`(スタティック・キャスト)
  * `const_cast`(コンスト・キャスト)
  * `dynamic_cast`(ダイナミック・キャスト)
  * `reinterpert_cast`(リインタープリト・キャスト)
* `static_cast`は次のように書く
  ```cpp
  size_t a = 123;
  int b = static_cast<int>(a);
  ```
* `dynamic_cast`を使うと、「基底クラスのポインタが指している実際のクラス」をチェックできる
* `reinterpret_cast`を使ってはいけない。


---

## <font face="BIZ UDゴシック">1 プリプロセッサ</font>

---


### 1.1 ビルド工程

プログラムのビルドは、実際には以下の4つの工程で構成されています。

&emsp;**①プリプロセス**&emsp;C++ファイルからコメントを削除し、プリプロセッサ指令を処理する<br>
&emsp;**②コンパイル**&emsp;&emsp;プリプロセス後のC++ファイルから、アセンブリ言語(※1)で書かれた「アセンブリ ファイル」を作成する<br>
&emsp;**③アセンブル**&emsp;&emsp;アセンブリ ファイルを、機械語(※2)で書かれた「オブジェクト ファイル」に変換する<br>
&emsp;**④リンク**&emsp;&emsp;&emsp;&emsp;複数のオブジェクト ファイルを結合して、「実行可能ファイル」を作成する

※1 アセンブリ言語: 機械語の各命令に、人間に理解しやすい短い名前を付けた言語<br>
※2 機械語: CPUが直接実行できる言語で、0と1の2つの数値だけで書かれる

$
\boxed{
\boxed{ C++ファイル　　　 } \\
　　　↓ \\
　①プリプロセス → \boxed{ プリプロセス済みC++ファイル } \\
　　　　　　　　　　　　　　↓ \\
　　　　　　　　　　　 ②コンパイル → \boxed{ アセンブリファイル } \\
　　　　　　　　　　　　　　　　　　　　　　↓ \\
　　　　　　　　　　　　　　　　　　　③アセンブル → \boxed{ オブジェクトファイル } \\
　　　　　　　　　　　　　　　　　　　　　　　　　　　　　　↓ \\
　　　　　　　　　　　　　　　　　　　　　　　　　　　　④リンク → \boxed{ 実行可能ファイル }　　　　　　　　　　　　　　　　　　　　　}
$

「プリプロセッサ」は、この図の「①プリプロセス」を行うツールです。

プリプロセッサは、初期のCコンパイラの機能不足を補うために導入されました。ですが、2026年現在のC++では、「プリプロセッサにはできるだけ頼らない」ことが推奨されています。現在のC++は、「かつてプリプロセッサに頼らないと解決できなかった問題」の大半を解決できるからです。

>`pre`(プリ)→「前」、`process`(プロセス)→「処理」、`-or`(アー)→「～する人、装置」より、<br>
>`preprocesser`(プリプロセッサ)→「前処理装置」


### 1.2 プリプロセッサ指令

プリプロセッサの機能は、`#`記号ではじまる「プリプロセッサ指令」によって制御します。<br>
プリプロセッサの主要な機能と、対応するプリプロセッサ指令は次のとおりです。

* 他のファイルに書かれたテキストを、指定された位置に挿入する(`#include`)
* テキストの一部を、別のテキストに置き換える(`#define`)
* 条件によって、ファイルの一部を残したり削除したりする(`#if`, `#ifdef`, `#else`, `#endif`)
* 字句を文字列リテラルに変換する(`#`演算子)
* 複数の文字列リテラルを結合して、ひとつの文字列リテラルにする(`##`演算子)


#### 1.2.1 プリプロセッサ指令は1行に1個まで

プリプロセッサ指令は、1行に1個だけ書くことができます。複数の指令を書く場合は行を分けます。

```cpp
#include "a.h" // OK.

#include "f.h" #include "g.h" // NG. 1行に1個だけ
```

プリプロセッサ指令の前後には、「空白」または「コメント」を入れられます。<br>
コメントにならない英数字や記号は書けません。

```cpp
   #   include    "a.h"   // OK. 空白は自由に書ける
/*コメント*/#include "c.h"/* OK. コメントは空白と同じ扱い */

 # i n c l u d e "b.h"      // NG. キーワードは空白で分けられない

//コメント   #include "d.h" // NG. //記号は行末までコメントになる

int a = 57; #include "e.h"  // NG. 空白にならない文字は書けない
```


#### 1.2.2 プリプロセッサ指令には`;`を書かない

プリプロセッサ指令は、最後に`;`記号を付けません。<br>
プリプロセッサ指令では、`;`記号ではなく「改行」を行の終わりとみなします。


### 1.3 ファイルのインクルード

`#include`指令は、引数で指定されたファイルのテキストを、`#include`行の直後に挿入します。

```cpp
#include <iostream>
#include "my_header.h"
```




#### 1.3.1 標準インクルード・ディレクトリ

引数を`<`と`>`で囲んだ場合、プリプロセッサはC++標準ライブラリのディレクトリからファイルを検索します。このディレクトリは「標準インクルード・ディレクトリ」や「システム・ディレクトリ」と呼ばれます。

>「ディレクトリ」は「フォルダ」の古い呼び方です(正確には違う部分があるのですが、気にするほどではないです)。

通常、「標準インクルードディレクトリ」は、Visual Studioなどの開発用アプリが自動的に設定します。

画像処理プログラムや3D処理プログラムなどを手動で追加した場合、それらのフォルダを標準インクルードディレクトリに追加することが多いです。追加方法はアプリごとに異なるので、ドキュメントを参照してください。


#### 1.3.2 ファイル・ディレクトリ

引数を`"`で囲んだ場合、プリプロセッサは処理中のファイルがあるディレクトリを検索します。このディレクトリは「ファイル・ディレクトリ」と呼ばれます。

指定されたファイルが「ファイル・ディレクトリ」で見つからなかった場合、標準インクルードディレクトリを検索します。

いずれの場合も、最後までファイルが見つからなかった場合はエラーとなります。


#### 1.3.3 テキストファイルならなんでもインクルードできる

`#include`で読み込めるファイルの条件は「テキスト形式であること」だけです。

また、`#include`を書く場所も決まっていません。そのため、次のように「テキストファイルを読み込んでデータとして使う」というプログラムも作成可能です。

**maze.txt**

```txt
 8, 8,
 1, 1, 1, 1, 1, 1, 1, 1,
 1, 0, 1, 0, 1, 0, 0, 1,
 1, 0, 1, 0, 1, 1, 0, 1,
 1, 0, 0, 0, 0, 1, 0, 1,
 1, 1, 1, 1, 0, 1, 0, 1,
 1, 0, 0, 1, 0, 1, 0, 1,
 1, 0, 1, 0, 0, 0, 0, 1,
 1, 1, 1, 1, 1, 1, 1, 1,
 ```

**main.cpp**

```cpp
#include <iostream>
using namespace std;

int main()
{
  const int maze[] = {
#include "maze.txt" // maze.txtをインクルード
  };

  const int width = maze[0];
  const int height = maze[1];
  const char c[] = { '.', '#' };
  for (int y = 0; y < height; y++) {
    for (int x = 0; x < width; x++) {
      const int type = maze[y * width + x + 2];
      cout << c[type];
    }
    cout << endl;
  }
}
```

**実行結果**<br>
<font face="Noto Sans">&emsp;########<br>
&emsp;#.#.#..#<br>
&emsp;#.#.##.#<br>
&emsp;#....#.#<br>
&emsp;####.#.#<br>
&emsp;#..#.#.#<br>
&emsp;#.#....#<br>
&emsp;########</font>

実際のアプリでは、`fstream`などのファイル操作クラスを使って、実行時にファイルを読み込むほうがよいでしょう。

`#include`を使う方法では、データファイルを変更するたびに毎回ビルドし直す必要があるからです。実行時に読み込む方法ならデータファイルを変更してもビルド不要なので、変更の確認を素早く行えます。

>滅多に変更しないファイルであれば`#include`で読み込むのもありですが、あまりお勧めはしません。

### 1.4 マクロ

>マクロはできるだけ使わないようにしましょう。<br>
>マクロはC++とは異なる動作をするため理解しにくく、動作を予想しづらいためです。

`define`指令は、ファイル内で「元のテキスト」を見つけたら、「置き換え先のテキスト」で置き換えるように指示します。

```cpp
#define 元のテキスト 置き換え先のテキスト
```

この「テキスト置き換え機能」のことを「マクロ」といいます。

`define`は次のように使います。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC DEF
#define DEF 123
#define X   cout << 456 << endl;

int main() {
  cout << ABC << endl; // ABC はまず「DEF」に置き換えられ、そして DEF は「123」に置き換えられる
  cout << DEF << endl; // DEF は「123」に置き換えられる
  X                    // X は「cout << 456 << endl;」に置き換えられる
}
```

**実行結果**

```txt
123
123
456
```

>マクロもプリプロセッサ指令なので、行末の`;`記号は不要です。

上記のプログラムでは、以下のようにテキスト置換が行わます。

1. 文章`ABC`はマクロ`ABC`と一致するので、文章`DEF`に置き換えられる
2. 文章`DEF`はマクロ`DEF`と一致するので、文章`123`に置き換えられる
3. 文章`123`に対応するマクロはないので、テキスト置換を終了する

なお、マクロで置換された結果が別のマクロになる場合、「別のマクロ」の定義は前にあっても後ろにあっても構いません。マクロは、使われた時点ではじめてテキスト置換が行われるためです。使われるまでに定義されていれば、どこにあるかは問われません。

```cpp
#include <iostream>
using namespace std;

int main() {
  #define A B
  #define B 65
  cout << A << endl;   // OK. Bに変換され、Bはこの行より前に定義されている

  #define C D
  //cout << C << endl; // NG. D二変換されるが、Dはこの行より後で定義されている
  #define D 68
}



#### 1.4.1 マクロの名前は大文字にする

マクロの名前(マクロ名)のルールは、C++の名前のルールと同じです。<br>
つまり、「英文字か`_`で始まり、英数字か`_`が続く」というルールです。

そのため、マクロ名には小文字を含めることができます。

```cpp
#define abc 1 // OK
#define Abc 2 // OK
#define ABC 3 // OK
```

大文字と小文字は区別されるので、上記の3つのマクロ名は「すべて異なるもの」として扱われます。

ですが、マクロ名は伝統的に「英字は大文字だけ」で書くことになっています。<br>
主な理由は、

&emsp;**マクロにはスコープがないので、通常のC/C++と同じ名前ルールを使うとエラーやバグを起こす可能性がある**

というものです。例えば、以下のプログラムはエラーになります。

**コード**

```cpp
#include <iostream>
using namespace std;

#define index 3

int main() {
  int index = 0;
  cout << index << endl;
}
```

**実行結果**

```txt
test.cpp: In function ‘int main()’:
test.cpp:4:16: error: expected unqualified-id before numeric constant
    4 | # define index 3
      |                ^
test.cpp:7:7: note: in expansion of macro ‘index’
    7 |   int index = 0;
      |       ^~~~~
```

このエラーの意味は、<br>
&emsp;`int index = 0;`の`index`という名前が、`3`というテキストで置き換えられている<br>
というものです。

このように、マクロ名として「通常のC++で書かれそうな名前」を付けてしまうと、意図せずC++のプログラム自体が置き換えられてしまう危険性があるのです。

マクロ名をすべて大文字にすれば、このようなエラーを防げます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define INDEX 3

int main() {
  int index = 0;
  cout << index << endl;
}
```

**実行結果**<br>
&emsp;0


#### 1.4.2 `￥`を使って2行以上のプリプロセッサ指令を書く

行末に`￥`(円記号、バックスラッシュ)を書くと、次の行までプリプロセッサ指令を続けて書くことができます。

そのため、次のようにマクロの末尾に`￥`(円記号、バックスラッシュ)を書くと、見やすいマクロが書けます。

```cpp
#include <iostream>
using namespace std;

#define OUTPUT5 \
  cout << 1; \
  cout << 2; \
  cout << 3; \
  cout << 4; \
  cout << 5 << endl;

int main() {
  OUTPUT5
}
```

**実行結果**<br>
&emsp;1 2 3 4 5

なお、`￥`記号の実際の効果は「次の行を現在の行の後ろにつなぐ」というものです。<br>
そのため、上記の`OUTPUT5`マクロは、次のように書いたものとして扱われます。

```cpp
#define OUTPUT5 cout << 1;   cout << 2;   cout << 3;   cout << 4;   cout << 5 << endl;
```


#### 1.4.3 マクロの多重定義

マクロは、内容が完全に同じなら、同名のマクロを何回でも書けます。

```cpp
#define A 123
#define A 123   // OK. 前の定義と同じ
//#define A 124 // NG. 前の定義と違う
#define B 124   // OK. 違うマクロ
```


### 1.5 定義済みマクロ

C++では、特別な機能を持つ次のマクロが自動的に定義されます。<br>
これらはプログラムのどこでも使えます。

* `__FILE__`&emsp;現在のファイル名に置き換えられるマクロ
* `__LINE__`&emsp;現在の行番号に置き換えられるマクロ
* `__DATE__`&emsp;プリプロセスを実行した年月日に置き換えられるマクロ
* `__TIME__`&emsp;プリプロセスを実行した時刻に置き換えられるマクロ

これらのマクロは、主に実行時エラーや論理エラーを検出するコードにおいて、検出位置や日時をコンソールウィンドウに表示するために使われます。

>これらのマクロとよく似たものに`__func__`という「定義済みローカル変数」があります。<br>
>`__func__`は関数の名前を指すポインタになっています。

**コード**

```cpp
#include <iostream>
using namespace std;

int main() {
  cout << "__FILE__: " << __FILE__ << endl;
  cout << "__LINE__: " << __LINE__ << endl;
  cout << "__DATE__: " << __DATE__ << endl;
  cout << "__TIME__: " << __TIME__ << endl;
  cout << "__func__: " << __func__ << endl;
}
```

**実行結果**<br>
&emsp;\_\_FILE__: main.cpp<br>
&emsp;\_\_LINE__: 6<br>
&emsp;\_\_DATE__: Jun 11 2026<br>
&emsp;\_\_TIME__: 07:38:07<br>
&emsp;\_\_func__: main

>`__FILE__`, `__LINE__`, `__func__`の3つに関しては、C++20で追加された`source_location`(ソース・ロケーション)構造体で置き換えられます。


### 1.6 関数型マクロ

`define`指令は、次のような引数付きのマクロを定義することもできます。<br>
このような引数付きのマクロは、「関数型マクロ」と呼ばれます。

```cpp
#define 元のテキスト(引数のリスト) 置き換え先のテキスト
```

「置き換え先のテキスト」に引数を書くと、引数として渡されたテキストに置き換えられます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define MAKE_FUNC(name, op)  int name(int a, int b) { return a op b; }

MAKE_FUNC(Plus, +)     // int Plus(int a, int b) { return a + b; } に置き換えられる
MAKE_FUNC(Multiply, *) // int Multiply(int a, int b) { return a * b; } に置き換えられる

int main() {
  cout << Plus(1, 2) << endl;
  cout << Multiply(3, 4) << endl;
}
```

**実行結果**

```
3
12
```

上のプログラムの`MAKE_FUNC`(メイク・ファンク)マクロは、2つの値を引数`op`で指定した方法で処理する関数を、`name`で指定した名前で作成します。

このように、関数型マクロは、置き換え先のテキストの引数を、引数として与えられたテキストで置き換えます。


#### 1.6.1 マクロ名と括弧の間には空白を入れない

関数型マクロでは、「マクロ名」と`(`記号の間に空白を入れてはいけません。<br>
空白を入れると、関数型ではない「普通のマクロ」になってしまうからです。

```cpp
#define A(a) a * 2 // 空白がない → 「a * 2」に置換される関数型マクロ

#define B (b) b * 2 // 空白がある → 「(b) b * 2」に置換される普通のマクロ
```


#### 1.6.2 関数型マクロと定義済みマクロを組み合わせる

関数型マクロと定義済みマクロを組み合わせると、デバッグ用のメッセージを出力するマクロを作れます。

以下の`DEBUG_LOG`マクロは、メッセージの前にファイル名と行番号を追加して出力します。

**コード**

```cpp
#include <iostream>
#include <fstream>
using namespace std;

#define DEBUG_LOG(text) \
  { cout << "LOG in " << __FILE__ << ":" << __LINE__ << ": " << text << endl; }

int main() {
  fstream file("test.txt");
  if ( ! file) {
    DEBUG_LOG("ファイルが開けません");
  } else {
    // fileを操作するプログラム
  }
}
```

**実行結果**<br>
&emsp;LOG in test.cpp:10:ファイルが開けません


#### 1.6.3 関数型マクロの引数はカッコで囲もう

「関数型マクロ」はC++言語の関数と似ていますが、機能的には「テキストを置き換えるだけ」です。<br>
そのため、`+`や`/`などの記号も引数に渡せます。

次のプログラムは、C++の関数と関数型マクロの違いを示す例です。

**コード**

```cpp
#include <iostream>
using namespace std;

#define A(a)  a * a

int B(int a) { return a * a; }

int main() {
  cout << A(1 + 2) << endl; // A(1 + 2) は 1 + 2 * 1 + 2 に置き換えられる
  cout << B(1 + 2) << endl;
}
```

**実行結果**

```txt
5
9
```

関数型マクロがやることは、引数を展開してテキストを置き換えるだけです。そのため、`A(1 + 2)`というマクロ呼び出しは、`a`を`1 + 2`に置き換えます。

その結果、`a * a`が`1 + 2 * 1 + 2`に置き換わります。演算子の優先順位により`2 * 1`が先に計算されるため、`5`が出力されるのです。

これに対して、関数の場合は呼び出し前に計算が行われます。つまり、関数`B`より前に`1 + 2`が計算されて、引数`a`には`3`が代入されるわけです。その結果、`9`が出力されるのです。

関数型マクロでも関数と同じ結果を得たい場合、次のように括弧を使って優先順位を制御します。

```cpp
#define A(a) (a) * (a)
```

マクロをこのように定義すると、`A(1 + 2)`は`(1 + 2) * (1 + 2)`に置き換えられます。これなら、優先順位の問題は回避できます。


#### 1.6.4 関数型マクロは関数ではない

マクロ引数に関数が指定された場合、括弧を使っても問題を回避できない場合があります。

次のプログラムは、マクロ`A`と関数`B`のそれぞれに対して、関数`func`を指定した場合の違いを示しています。

**コード**

```cpp
#include <iostream>
using namespace std;

int g = 0;
int func() { g++; return g; }

#define A(a) (a)+(a)

int B(int b) { return b + b; }

int main() {
  cout << A(func()) << endl; // (func())+(func()) になる
  cout << B(func()) << endl; // int b = func(); B(b); になる
}
```

**実行結果**<br>
&emsp;3<br>
&emsp;6

マクロ`A(func())`は`(func())+(func())`に展開されます。<br>
そのため、`func`が2回呼び出されて、それぞれの結果は`1`と`2`になります。<br>
そして、`1 + 2`が計算されて`3`が出力されるのです。

これに対して関数`B(func())`の場合、`func()`が先に実行されて`3`になり、その後、関数`B`の引数`b`に`3`が代入されます。<br>
これを受けて関数は`3 + 3`を計算するので、結果が`6`になるわけです。

このように、関数型マクロは関数のように見えますが、本来の目的は「テキストの置換」で、関数とは全く異なります。<br>
関数のように使ってしまうとバグやエラーの原因になりかねません。

>2026年現在のC++では、関数型マクロが必要となる場面はほとんどありません。できるだけ使わないようにしましょう。


#### 1.6.5 可変引数マクロ

「可変引数(かへん・ひきすう)マクロ」は、関数型マクロの引数の数を、使う場面によって変えられる機能です。<br>
引数の名前の代わりに`...`(ドット3個)を書くことで、0個以上の任意の個数の引数を受け取れます。

受け取った引数は、`__VA_ARGS__`(バ・アーグス)というキーワードで参照できます。

**コード**

```cpp
#include <iostream>
#include <vector>
using namespace std;

#define A(...) __VA_ARGS__
#define V(b, ...) vector<string> b = { __VA_ARGS__ }

int main() {
  vector<int> v = { A(1, 2, 3) };
  V(s, "aaa", "bbb", "ccc", "ddd");

  for (auto i = v.begin(); i != v.end(); i++) {
    cout << *i << ' ';
  }

  for (auto i = s.begin(), i != s.end(); i++) {
    cout << *i << ' ';
  }
  cout << endl;
}
```

**実行結果**<br>
&emsp;1 2 3 aaa bbb ccc ddd

>可変引数マクロは、デバッグ出力用のマクロを作るときや、高度なライブラリを作成する補助ツールとして使われています。しかし、ほとんどのアプリ開発では使う必要がありません。<br>
>可変長引数マクロもマクロの仲間なので、できるだけ使わないほうがいいです。


### 1.7 マクロの削除

`undef`(アンデフ)指令は、マクロの定義を削除します。

```cpp
#undef 削除したいマクロ名
```

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC  123
#define X(a) (a)+(a)

#undef ABC       // マクロ ABC を未定義にする
#undef X         // マクロ X を未定義にする

int main() {
  cout << ABC << endl;   // エラー. ABC は未定義
  cout << X(57) << endl; // エラー. X は未定義
}
```

`undef`指令は、定義を変更したい場合にも使われます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC 123
#undef ABC       // マクロ ABC を未定義にする
#define ABC 456  // マクロ ABC を定義しなおす

int main() {
  cout << ABC << endl; // 定義があるのでエラーにはならない
}
```

**実行結果**

```txt
456
```

なお、未定義のマクロに対して`undef`指令を使ってもエラーにはなりません。

**コード**

```cpp
#include <iostream>
using namespace std;

#undef ABC // OK.

#define ABC "macro"

int main() {
  cout << ABC << endl;
```

**実行結果**

```txt
macro
```


### 1.8 テキストの文字列化

関数型マクロでは、`#`と`##`という2つの演算子が使えます。

`#`演算子はテキストを「文字列化」します。

**コード**

```cpp
#include <iostream>
using namespace std;

#define STR(a) #a

int main() {
  cout << STR(this is a "pen".) << endl; // "this is a \"pen\"." で置き換えられる
}
```

**実行結果**

```txt
this is a "pen".
```

> `#`演算子は、元のテキストに含まれる`"`記号を`\"`に変換します。

「文字列化」の注意点として、「引数にマクロを指定しても、置き換えられずにそのまま文字列化される」ということがあります。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC 123

#define STR(a) #a

int main() {
  cout << STR(ABC) << endl;
}
```

**実行結果**

```txt
ABC
```

このように、マクロ`ABC`は`123`に置き換えられず、文字列`ABC`になってしまいます。

マクロを置き換えつつ文字列化するには、次のように２段階の関数型マクロを使います。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC 123

#define STR(a) #a
#define STR2(a) STR(a)

int main() {
  cout << STR2(ABC) << endl;
}
```

**実行結果**

```txt
123
```

マクロ`STR2(ABC)`は`STR(123)`に置き換えられるので、文字列`123`が出力されます。


### 1.9 テキストの連結

`##`演算子は、テキストを「連結」します(文字列化はしません)。

**コード**

```cpp
#include <iostream>
using namespace std;

#define CONCAT(a, b) a ## b

int main() {
  int test01 = 100;
  cout << CONCAT(test, 01) << endl; //  test01 で置き換えられる
}
```

**実行結果**

```txt
100
```

テンプレートが十分に強力になった2026年現在のC++において、テキストの連結を使う機会はほとんどありません。

`##`の使用例として、次のような「状態に対応する関数を実行し、実行前に状態名を出力する」プログラムを考えます。

```cpp
#include <iostream>
using namespace std;

void TitleFunc();
void MainFunc();
void GoalFunc();
void GameOverFunc();

enum class State { Title, Main, Goal, GameOver, Exit };
State state = State::Title;

int main()
{
  while (state != State::Exit) {
    switch (state) {
      case State::Title: cout << "Title\n"; TitleFunc(); break;
      case State::Main: cout << "Main\n"; MainFunc(); break;
      case State::Goal: cout << "Goal\n"; GoalFunc(); break;
      case State::GameOver: cout << "GameOver\n"; GameOverFunc(); break;
    }
  }
}
```

この例では、各 case 文において、`Title`などの状態名が3回ずつ現れます。プログラミングにおいて、同じことを3回も繰り返すのは悪いことです。この例は、マクロと`#`、それから`##`を使うと、次のように書き換えられます。

```cpp
#include <iostream>
using namespace std;

// case文を生成する関数型マクロ
#define CASE(s) case State::s: cout << #s "\n"; s ## Func(); break

void TitleFunc();
void MainFunc();
void GoalFunc();
void GameOverFunc();

enum class State { Title, Main, Goal, GameOver, Exit };
State state = State::Title;

int main()
{
  while (state != State::Exit) {
    switch (state) {
      CASE(Title);
      CASE(Main);
      CASE(Goal);
      CASE(GameOver);
    }
  }
}
```

`switch`文の内容がすっきりして、状態名はただ1度だけ書かれている点に注目してください。この例の`CASE`マクロは、`#s`と書くことで状態名を文字列に変換したり、`s ## Func();`と書くことで、`状態名Func();`という関数名を作り出しています。

また、`State::s:`のように前後の文字が記号や空白の場合、`##`は不要です。


#### 1.9.1 連結するとマクロになる場合

テキストを連結した結果が、何らかのマクロの名前に一致した場合、それはマクロとして扱われます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define TEST1 123
#define TEST2 "abc"
#define X(a) TEST ## a

int main() {
  cout << X(1) << endl;
  cout << X(2) << endl;
}
```

**実行結果**<br>
&emsp;123<br>
&emsp;abc

マクロ`X(1)`は`##`によって`TEST1`になります。そして、`TEST1`は存在するマクロ名なので、`123`に置き換えられます。

同様に、マクロ`X(2)`は`TEST2`に置き換えられ、`TEST2`は存在するマクロ名なので、`"abc"`に置き換えられます。

このように、マクロや`#`、`##`記号を使いこなすと、C/C++の機能だけでは実現できないプログラムを書くことができます。

>**【マクロと関数の実行順序】**<br>
>関数型マクロは「一番外側」から順番に置換されます(通常の関数は「一番内側」から順番)。<br>
>&emsp;`X(Y(Z(123)))`<br>
>というプログラムを書いた場合、まずマクロ`X`がテキスト`Y(Z(123))`を引数にして置換を行います。次にマクロ`Y`がテキスト`Z(123)`を引数にして置換を行い、最後にマクロ`Z`がテキスト`123`を引数にして置換を行います。


### 1.10 関数型マクロを多用しない

関数型マクロには、C/C++の制限を飛び越えて、より自由なプログラムを実現する能力があります。<br>
問題は、自由すぎるプログラムは「ひと目見ただけでは、何をしているのかが分かりにくい」となりがちなことです。

上記のいくつかの例でも、「誰が見てもすぐにやっていることが分かる」という点で比較すると、マクロを使わない書きかたのほうが優れています。

>**【可読性(かどくせい)】**<br>
>「誰が見てもすぐに内容が分かる」ことを「可読性が高い」と表現します。<br>
>とくに複数人で開発をしている場合、可読性が高いことは重要です。マクロを使って多少の手間を省いたとしても、それで可読性が下がってしまうと、あとでプログラムを修正する人の負担になってしまうからです。
>
>ここまで長々とマクロの説明しておいて申し訳ないのですが、マクロの使用は最小限にしましょう。


### 1.11 条件付きコンパイル

`#if`(イフ)指令、`#endif`(エンド・イフ)指令、`#else`(エルス)指令、`#elif`(エル・イフ)指令は、条件式によってプログラムの一部を残すか、削除するかを決定します。

プリプロセッサの条件式では、C++と同じように計算式や比較、論理演算が可能です。比較演算子と論理演算子の機能と評価順は、C++と同じです。<br>
ただし、C++の変数は使えません。使えるのは「マクロ」と「整数」だけです。<br>
条件式内でマクロを実行した結果が、マクロでも整数でもない文字列になった場合、その文字列は整数の`0`として扱われます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC 1

int main()
{
#if ABC > 0
  cout << "ABC は 0 より大きい" << endl;
#else
  cout << "ABC は 0 以下" << endl;
#endif

  int XYZ = 1;

// 変数は、プリプロセッサ指令の中では使えない
#if XYZ
  cout << "XYZ は 0 以外" << endl;
#else
  cout << "XYZ は 0 または未定義" << endl;
#endif
}
```

**実行結果**

```txt
ABC は 0 より大きい
XYZ は 0 または未定義
```

プリプロセッサで計算が実行されるのは「条件式に限られる」ことに注意して下さい。<br>

```cpp
#include <iostream>
using namespace std;

#define A 1
#define B 2
#define C(X) #X

int main() {
  cout << C(A + B) << endl;
}
```

**実行結果**

```txt
1 + 2
```



#### 1.11.1 defined演算子

`defined`(ディファインド)演算子は、マクロが定義されているかどうかを判定します。<br>
`defined`演算子はプリプロセッサの条件式でのみ使えます。

**コード**

```cpp
#include <iostream>
using namespace std;

#define ABC

int main()
{
#if defined XYZ
  cout << "xyz" << endl;
#else
#if defined ABC
  cout << "abc" << endl;
#else
  cout << "error" << endl;
#endif
#endif

}
```

**実行結果**

```txt
abc
```

また、`#if`と`defined`をくっつけた`#ifdef`(イフ・デフ)指令や、`#if`と`!defined`をくっつけた`#ifndef`(イフ・エヌ・デフ)指令を使うこともできます。

>さらに C++23 では、`#elif`と`define`、`!define`をくっつけた`#elifdef`、`#elifndef`が追加されました。<br>
>(この記事執筆時点(2026年6月)では、Google Colabのコンパイラは C++23 に部分的にしか対応していないため、`#elifdef`、`#elifndef`は使えません)。

**defined の代わりに #ifdef を使ったコード**

```cpp
#include <iostream>
using namespace std;

#define ABC

int main()
{
#ifdef XYZ
  cout << "xyz" << endl;
#else
#ifdef ABC
  cout << "abc" << endl;
#else
  cout << "error" << endl;
#endif
#endif
}
```

**実行結果**

```txt
abc
```


#### 1.11.2 ビルド環境固有の定義済みマクロ

プログラムをさまざまなビルド環境でビルドできるようにするには、ビルド環境を識別できなくてはなりません。<br>
この目的のために、ビルド環境ごとに固有の定義済みマクロが存在します。

例えば、Visual Studioには以下の定義済みマクロがあります。

* _MSC_VER: Visual Studio のコンパイラのバージョン番号として定義されます。
* _DEBUG: ソリューション構成が`Debug`の場合に定義されます。
* _WIN32: プラットフォームが`Win32`の場合に定義されます。
* _WIN64: プラットフォームが`x64`の場合に定義されます。

これらはVisual Studioが定義するマクロのほんの一部です。実際には数多くのマクロが定義されます。<br>
詳しくは以下のURLを参照して下さい。<br>
&emsp;<https://learn.microsoft.com/ja-jp/cpp/preprocessor/predefined-macros>

特に`_DEBUG`は、デバッグ中のみ実行したいプログラムを作る場合に重宝します。<br>
例としては、動作確認用の情報の出力や、データの正しさのチェックなどがあります。

>アプリ固有のデバッグ機能を作る場合は、そのアプリ専用のマクロを定義したほうがよいでしょう。


### プリプロセッサの練習問題

&emsp;[問題１ 配列を取り込む](#scrollTo=RnTTS8R1APhR)<br>
&emsp;[問題２ 角度の単位](#scrollTo=gTvMA7DPAFEK)<br>
&emsp;[問題３ 繰り返しマクロ](#scrollTo=B7ilKDxIqTEY)<br>
&emsp;[問題４ バージョン２のアプリ](#scrollTo=hytpYptiqToJ)


---

## <font face="BIZ UDゴシック">2 属性(アトリビュート)</font>

---


### 2.1 属性の役割

属性(アトリビュート)は、

&emsp;**より効率的で安全なアプリを作成するために、コンパイラなどのビルド環境に対して追加情報を伝える**

機能です。

属性は角括弧`[`と`]`を連続で2つ重ねて、次のように書きます。

```cpp
  [[ 属性の名前 ]]
```

主要な属性を次に示します。

| 属性の名前 | 対象 | 機能 |
|:-----------|:-----|:-----|
| <font size=3>noreturn</font><br>(ノー・リターン) | 関数 |<font size=3>関数が戻らないことを示す</font> |
| <font size=3>fallthrough</font><br>(フォール・スルー) | switch文 | <font size=3>`switch`文で意図的に`break`を書いていないことを示す</font> |
| <font size=3>maybe_unused</font><br>(メイビー・アンユーズド) | 変数,関数,型 | <font size=3>未使用変数の警告を出さないようにする</font> |
| <font size=3>nodiscard</font><br>(ノー・ディスカード) | 関数,型 | <font size=3>関数の戻り値を無視した場合に警告を出す</font> |
| <font size=3>deprecated</font><br>(デプリケーテッド) | 変数,関数,型 | <font size=3>非推奨の(近い将来削除されるであろう)関数であることを示す</font> |

以下では、各属性の機能を説明します。



### 2.2 noreturn 属性

関数に`noreturn`(ノー・リターン)属性を付けると、「関数から決して戻らない」ことを示します。

エラー処理を行った後でアプリを終了させる関数などで使います。

例えば、とても重要なファイルの読み込みに失敗して、それ以上アプリの実行を続けられない、という場合を考えます。

この場合、エラーメッセージを表示してアプリを終了することになるでしょう。<br>
次のプログラムは、`noreturn`属性を使った例です。

**コード**

```cpp
#include <iostream>
#include <stdlib.h>
using namespace std;

[[noreturn]] void report_and_exit(const char* message)
{
  cout << message << endl;
  exit(1); // ここでプログラムが終了する(リターンしない)
}

int div(int a, int b)
{
  if (b == 0) {
    report_and_exit("ゼロ除算が発生しました");
  }
  return a / b;
}

int main()
{
  cout << div(10, 0) << endl;
}
```

**実行結果**

```txt
ゼロ除算が発生しました
```

`noreturn`属性は、「決して戻らない」関数にしか付けられない点に注意して下さい。<br>
`if`文などで戻る可能性がある関数に付けてしまうと、コンパイルエラーが発生します。


### 2.3 fallthrough属性

`fallthrough`(フォール・スルー)属性は、`switch`文の中でのみ使える属性で、 **意図的に break を書かなかった** ことを示します。<br>
`fall through`(フォール・スルー)は「落ちる」、「失敗する」という意味です。プログラミング言語では、「`switch`文において、すぐ下にある`case`に進む」ことを意味します(下の`case`に「落ちる」というイメージ)。

次のプログラムは、`fallthrough`を書ける場所と、書けない場所の例です。

>`[[fallthrough]]` のあとに、`;`記号が必要な点に注意して下さい。

**コード**

```cpp
#include <iostream>
using namespace std;

int main() {
  int n = 3, m = 4;
  switch (n) {
  case 1:
    // OK. case が空なら警告されない(何も書かなくてよい)
  case 2:
    cout << "case 2" << endl;

  case 3: // NG. 上に break か [[fallthrough]] が必要
    cout << "case 3" << endl;
    [[fallthrough]];

  case 4: // OK. 上の [[fallthrough]] の効果で警告されない
    if (m > 4) {
      cout << "case 4 with big m" << endl;
      [[fallthrough]]; // OK. すぐに次の case に進める
    } else {
      [[fallthrough]]; // NG. 次の case に進む前にコードがある
      cout << "case 4" << endl;
    }

  case 5:
    cout << "case 5" << endl;
    [[fallthrough]]; // NG. 最後の case には書けない
  }
}
```

**実行結果**

```txt
main.cpp:10:25: warning: this statement may fall through [-Wimplicit-fallthrough=]
   10 |     cout << "case 2" << endl;
      |                         ^~~~
main.cpp:22:27: warning: this statement may fall through [-Wimplicit-fallthrough=]
   22 |       cout << "case 4" << endl;
      |                           ^~~~
main.cpp:21:22: warning: attribute ‘fallthrough’ not preceding a case label or default label
   21 |       [[fallthrough]]; // NG. 次の case に進む前にコードがある
      |                      ^
main.cpp:27:20: warning: attribute ‘fallthrough’ not preceding a case label or default label
   27 |     [[fallthrough]]; // NG. 最後の case には書けない
      |                    ^
case 3
case 4
case 5
```

上記の`case 4`のように、複数の`[[fallthrough]]`を書くとエラーを起こしやすいです。以下のように、`case`の最後に1回だけ書くほうがよいでしょう。

```c++
  case 4: // OK. 上の [[fallthrough]] の効果で警告されない
    if (m > 4) {
      cout << "case 4 with big m" << endl;
    } else {
      cout << "case 4" << endl;
    }
    [[fallthrough]]; // caseの最後に書く
  
  case 5:
```

>**【fallthrough属性の古い書き方】**<br>
>フォールスルーという機能は、C++11で属性が追加されるより前から利用できました。<br>
>やり方は簡単で、コメントとして、
>```cpp
>// fallthrough
>```
>または、
>```cpp
>/* fallthrough */
>```
>と書くだけです。この方法は、2025年現在でも多くのビルド環境で機能します。


### 2.4 maybe_unused属性

変数や関数、型に`maybe_unused`(メイビー・アンユーズド、「多分使わない」という意味)属性を付けると、その名前を使わない場合の警告が無効化されます。

プログラミングにおいて、デバッグ中にだけ使われる変数や関数、引数を宣言したり、定義したい場合があります。

そのような変数や関数があるプログラムを、デバッグ以外の設定でコンパイルすると、コンパイラは「この変数や関数は使われていない」という警告を出力します。

警告はプログラムの問題を見つけるために有用ですが、この場合は無意味です。<br>
そんなときは、`maybe_unused`属性を付けます。すると、無意味な警告を消せます。

次のプログラムは、`maybe_unused`属性を使った例です。

**コード**

```cpp
#define NDEBUG
#include <iostream>
#include <assert.h>
using namespace std;

// 警告されるかもしれない(コンパイラや設定による)
void MightGetWarned(bool a, bool b) {
  bool c = a;
  assert(b && c);
}

// 警告されない
void WontGetWarned(bool a, [[maybe_unused]] bool b) {
  [[maybe_unused]] bool c = a;
  assert(b && c);
}

int main() {
  MightGetWarned(true, true);
  WontGetWarned(true, true);
}
```

**実行結果**

```txt
main.cpp: In function ‘void might_get_warned(bool, bool)’:
main.cpp:8:8: warning: unused variable ‘c’ [-Wunused-variable]
    8 |   bool c = a;
      |        ^
main.cpp:7:36: warning: unused parameter ‘b’ [-Wunused-parameter]
    7 | void MightGetWarned(bool a, bool b) {
      |                               ~~~~~^
```

プログラムの先頭にある`#define NDEBUG`によって、`assert`マクロが無効化される点に注意して下さい。

そのため、`MightGetWarnd`(マイト・ゲット・ワーンド、「警告されるだろう」)関数では、引数`b`とローカル変数`c`が未使用になります。<br>
実行結果ではこの2つについて警告が表示されています。

対して、`WontGetWarned`(ウォント・ゲット・ワーンド、「警告されないだろう」)関数では、引数`b`とローカル変数`c`に`[[maybe_unused]]`を付けました。<br>
これによって、これらが未使用であっても警告はされなくなります。

なお、先頭の`#define NDEBUG`の行を削除すると、`MightGetWarnd`関数でも警告は出ません。

>**【maybe_unused属性を付けられる場所】**<br>
>`maybe_unused`属性は、引数や変数だけでなく、構造体、クラス、関数などにも付けられます。主な用途は同じで、デバッグ専用のクラスや関数のように、コンパイル時の設定などの「条件によって使われない場合がある」ものや、「開発が進んで使われなくなったもの」に付けます。


### 2.5 nodiscard属性

関数または型に`nodiscard`(ノー・ディスカード)属性を付けると、関数の戻り値を無視したときに警告されます。

実際のところ、大抵の戻り値は重要データなので、滅多に無視されることはありません。<br>
ですが、戻り値が補助的に使われる、例えば、エラー状態を返すような関数では、戻り値が無視される場合がありえます。

理想的には、戻り値を`optional`型に変えるなどで改善するべきです。しかし、関数がすでにあちこちで利用されている場合はそうもいきません。変更箇所が多くなるほど、プログラムを変更して動作テストするのにかかる時間が増えるからです。

この場合、関数に`nodiscard`属性を付ければ、他のプログラムを一切変更することなく、エラー状態を無視している場所を見つけられます。

**コード**

```cpp
#include <iostream>
#include <string>
using namespace std;

// 無視すると警告される型(構造体の場合、 nodiscard は struct と型名の間に書く)
struct [[nodiscard]] Result {
  bool isSuccess;
  string errorMessage;
};

// 無視すると警告される型を返す関数
Result PrintDiv(int a, int b) {
  if (b == 0) {
    return { false, "0除算エラー" };
  }

  if (a % b) {
    cout << a << "/" << b << "=" << a / b << "..." << a % b << endl;
  } else {
    cout << a << "/" << b << "=" << a / b << endl;
  }
  return { true, {} };
}

// 戻り値を無視すると警告される関数(関数の場合、 nodiscard は先頭に書く)
[[nodiscard]] int calc(int x, int y) { return x ^ y; }

int main() {
  Result r = PrintDiv(14, 0);// OK. 戻り値の Result 型を使っている
  if (!r.isSuccess) {
    cout << r.errorMessage << endl;
  }

  int a = calc(123, 456); // OK. 戻り値を使っている
  cout << a << endl;

  PrintDiv(10, 3); // NG. 無視してはいけない型を無視した
  calc(4, 5); // NG. 戻り値を無視した
}
```

**実行結果**

```txt
main.cpp:37:18: warning: ignoring returned value of type ‘Result’, declared with attribute ‘nodiscard’ [-Wunused-result]
   37 |   PrintDiv(10, 3); // NG. 無視してはいけない型を無視した
      |                  ^
main.cpp:38:7: warning: ignoring return value of ‘int calc(int, int)’, declared with attribute ‘nodiscard’ [-Wunused-result]
   38 |   calc(4, 5); // NG. 戻り値を無視した
      |   ~~~~^~~~~~
0除算エラー
435
10/3=3...1
```


### 2.6 deprecated属性

プログラムの開発が進んだ結果、最初に作った関数や構造体では問題が合ったり、機能が不足する、ということはよくあります。<br>
そのような場合、改良されたより良い関数や構造体を作ることになります。

しかし、`nodiscard`属性でも説明したように、いきなり古い機能を捨てて、すべてを新しい機能に置き換えるのは難しい場合があります。そんなときは、古い機能に`deprecated`(デプリケーテッド)属性を付けます。

`deprecated`(デプリケーテッド)は「非推奨(ひすいしょう)」という意味で、この属性が付いた機能を使うと警告メッセージが出力されます。

`deprecated`属性には2つの書き方があります。ひとつは通常の属性の書き方で、次のように書きます。

```cpp
  [[deprecated]]
```

もうひとつは、次のように括弧の中に文字列を入れる書き方です。<br>
文字列には「非推奨となった理由」や「代わりに使う機能」を書きます。

```cpp
  [[deprecated("追加のオプションを指定できるNewClassを使うこと")]]
```

括弧の中に書いた文字列は、警告メッセージの一部として出力されます。<br>
警告された原因と対策が分かるので、可能な限りこちらの書き方を使うとよいでしょう。

次のプログラムは、`deprecated`属性を使った例です。

**コード**

```cpp
#include <iostream>
using namespace std;

[[deprecated]] void Func() {
  cout << "この関数は十分に機能的だと思う" << endl;
}

[[deprecated("この関数の代わりにFuncVer3を使って下さい")]]
void FuncVer2() {
  cout << "今度こそ十分に機能的だと思う" << endl;
}

void FuncVer3() {
  cout << "本当に今度こそ十分に機能的だと思う" << endl;
}

struct [[deprecated("16bitのidを持つItem2を使って下さい")]] Item {
  unsigned int price;
  unsigned char id;
  char name[27];
};

struct Item2 {
  unsigned int price;
  unsigned short id;
  char name[26];
};

int main()
{
  Func();
  FuncVer2();

  Item item = { 300, 2, "高級傷薬" };
  cout << item.price << endl;
}
```

**実行結果**

```txt
main.cpp:31:7: warning: ‘void Func()’ is deprecated [-Wdeprecated-declarations]
   31 |   Func();
      |   ~~~~^~
main.cpp:32:11: warning: ‘void FuncVer2()’ is deprecated: この関数の代わりにFuncVer3を使って下さい [-Wdeprecated-declarations]
   32 |   FuncVer2();
      |   ~~~~~~~~^~
main.cpp:34:8: warning: ‘Item’ is deprecated: 16bitのidを持つItem2を使って下さい [-Wdeprecated-declarations]
   34 |   Item item = { 300, 2, "高級傷薬" };
      |        ^~~~
この関数は十分に機能的だと思う
今度こそ十分に機能的だと思う
300
```

`Func`関数の例では、単に「'void Func()' is deprecated」と出力されるだけで、非推奨となった原因までは分かりません。<br>
「どうすれば良いか」を知るには、関数宣言のあるヘッダファイルや外部の文書を調べる必要があります。

`FuncVer2`関数や`Item`構造体の例では、`deprecated`属性に書いた文字列が一緒に出力されています。<br>
「どうすれば良いか」が分かるので、機能の利用者は対策を取りやすくなります。

>ここまで見てきたように、「属性」にはさまざまな種類があります。<br>
>すぐに使えそうな属性もあれば、あまり使う機会のない属性もあるでしょう。<br>
>とりあえず、「現代のC++には属性という機能がある」ということだけ覚えておいて下さい。


### 属性(アトリビュート)の練習問題

&emsp;[問題５ 移動攻撃](#scrollTo=hZQUeY7Bqj76)<br>
&emsp;[問題６ 使われないかもしれない](#scrollTo=qBB2-M-dqx3a)


---

## <font face="BIZ UDゴシック">3 名前空間(ネームスペース)</font>

---


### 3.1 名前空間の書き方

C++の「名前空間(なまえくうかん)」は、次の機能を実現するために作られました。

* 外部ライブラリなどの他のプログラムと組み合わせたときに、名前の重複を防ぐ
* プログラムを管理しやすくするために、目的別のグループに分割する

名前空間を定義するには、`namespace`(ネームスペース)キーワードを使って次のように書きます。

```cpp
namespace 名前空間名 {
  名前空間のメンバにしたい構造体や関数、変数などを好きなだけ書く
} // ここに ; は不要！
```

書き方は構造体の定義と似ていますが、以下の点で異なります。

* 同じ名前の名前空間を定義できる(それらは共通のスコープを持つ)
* `public`や`private`といったアクセス制御がない(すべて`public`扱い)
* 関数や構造体の中には書けない(名前空間の中に名前空間を書く、つまり「名前空間のネスト」はできる)
* 最後の`}`のあとに`;`を付けない

名前空間を囲む`{`から`}`までの範囲は、「名前空間スコープ」になります。名前空間スコープの中で定義した変数や関数、構造体は、特に何もせずともそのまま使えます。

名前空間の基本的な使いみちは

&emsp;**自分のプログラムは独自の名前空間に格納する**

ことです。例えば、自分のプログラム用の`Position`構造体と`Add`関数を、名前空間を使って次のように定義できます。

```cpp
namespace MyApp { // MyApp という名前の「名前空間」を定義し、スコープを開始

  struct Position { int x, y; };

  // スコープ内で定義した Position 構造体を使う
  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }

} // 名前空間のスコープは、最初の「{」記号に対応する「}」記号で終わる
```


#### 3.1.1 名前空間の名付けかた

名前空間に付ける名前は「アプリ名」か、「機能をあらわすフレーズ」が多いです。<br>
アプリ名やフレーズが長すぎる場合は適当に短縮します。ですが、あまり短いと、他のプログラムが使う名前空間と重複する可能性が高くなります。そのため、少なくとも3文字以上にします。

```cpp
// 「暗黒世界のお散歩シミュレータ」は他と重複しなさそうな名前だが、長い
namespace TheDarkestWorldWalkingSimulator {
}

// 頭文字を使って短くする. 検索した限りでは重複は少なそう
namespace TDWWS {
}

// WalkingSimulator 部分だけ短縮. 短すぎるので重複しそう
namespace WS {
}
```


### 3.2 ::(スコープ解決演算子)

名前空間スコープの「外」で、ある名前空間のメンバを使うには、`::`(コロン２個)記号を使って次のように書きます。

```cpp
名前空間名::メンバ名
```

この`::`記号は「スコープ解決演算子」といいます。<br>
次のプログラムは、スコープ解決演算子を使った例です。

**コード**

```cpp
#include <iostream>
using namespace std;

namespace MyApp { // MyApp という名前の「名前空間」を定義

  struct Position { int x, y; };

  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }
}

int main() {
  // 名前空間 MyApp のメンバを使う
  MyApp::Position a = { 1, 2 };
  MyApp::Position b = MyApp::Add(a.x, a.y, 3, 4);
  cout << b.x << ' ' << b.y << endl;
}
```

**実行結果**<br><font face="monospace">
&emsp;4 6</font>

名前空間がネストしている場合、ネストしているすべての名前を`::`でつなぎます。

**コード**

```cpp
#include <iostream>
using namespace std;

namespace MyApp {

  // MyApp::Position構造体
  struct Position { int x, y; };

  // MyApp::Add 関数
  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }

  namespace Collision {
    
    // MyApp::Collision::Overlap 関数
    bool Overlap(const Position& a, const Position& b) {
      return a.x == b.x && a.y == b.y;
    }

  }
}

int main() {
  MyApp::Position a = { 1, 2 };
  MyApp::Position b = { 3, 4 };

  // MyApp::Collision 名前空間のメンバを使う
  if (MyApp::Collision::Overlap(a, b) == true) {
    cout << "hit" << endl;
  } else {
    cout << "miss" << endl;
  }
}
```

**実行結果**<br><font face="monospace">
&emsp;miss</font>


#### 3.2.1 ADL(引数依存の名前探索)

ある関数と、その関数の引数の型が同じ名前空間に含まれている場合、関数の名前空間を省略できます。<br>
この仕組みを`ADL`(エーディーエル、Argument Dependent Lookup、「引数依存の名前探索」という意味)といいます。

次のプログラムは、`::`演算子のプログラム例を少し変更して、`Add`関数の引数を`Position`型に変更したバージョンを追加しています。

**コード**

```cpp
# include <iostream>
using namespace std;

namespace MyApp { // MyApp という名前の「名前空間」を定義

  struct Position { int x, y; };

  // 引数が int のバージョン
  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }

  // 引数が Position のバージョン
  Position Add(Position a, Position b) {
    Position c = { a.x + b.x, a.y + b.y };
    return c;
  }
}

int main() {
  MyApp::Position a = { 1, 2 };
  MyApp::Position b = { 3, 4 };
  MyApp::Position c;

  c = MyApp::Add(a.x, a.y, b.x, b.y); // MyApp:: が必要
  cout << c.x << ' ' << c.y << endl;

  c = Add(a, b); // MyApp:: は不要
  cout << c.x << ' ' << c.y << endl;
}
```

**実行結果**<br><font face="monospace">
&emsp;4 6<br>
&emsp;4 6</font>

2回目の`Add`関数呼び出しに、`MyApp::`を付けていない点に注目してください。この`Add`関数は引数に`MyApp::Position`型を受け取るバージョンです。<br>
そのため、`MyApp`名前空間にある`Add`関数は、明示的に名前空間を指定していなくても、使用可能だと判断されます。

>もともと`ADL`は、演算子オーバーロードのために導入されました。<br>
>ところが、いろいろと便利に使えることが判明したため、標準ライブラリの様々な場面で悪用されています。<br>
>`ADL`はバグを生みやすいので、可能な限り名前空間を明示するべきです。


### 3.3 std名前空間

名前空間の分かりやすい例は、C++の標準ライブラリです。<br>
C++標準ライブラリは、ほぼすべての機能が`std`(エスティーディー)名前空間の中で宣言、定義されています。

>`std`は「標準」を意味する英単語 <font color="#e84">st</font>andar<font color="#e84">d</font>(スタンダード)の短縮形です。

名前空間のおかげで、アプリのプログラムに標準ライブラリと同じ名前の関数や構造体を定義しても、多重定義エラーにはなりません。例えば、何かを検索する`find`(ファインド)という関数を作成したとします。このとき、標準ライブラリの同名の関数を使いたい場合は、`std::find`と書くことができます。

次のプログラムは、アプリ独自の`find`関数と、標準ライブラリの`find`関数を使い分ける例です。

```cpp
#include <vector>
#include <algorithm>

struct Data {
  int id;
};

Data* find(vector<Data>& v, int id);

int main() {
  vector<Data> a(100);
      ︙
  Data* p = find(a, 57); // アプリの find を使う

      ︙

  vector<int> b(23);
      ︙
  auto itr = std::find(b.begin(), b.end(), 33); // 標準ライブラリの find を使う
      ︙
}
```


### 3.4 グローバル名前空間

グローバル名前空間は「名前空間を指定しなかった場合のデフォルトの名前空間」です。C++プログラムのすべての要素は「グローバル名前空間」のメンバです。

グローバル名前空間のメンバを明示的に指定したい場合は、次のようにメンバ名の前に`::`演算子だけを書きます。

```cpp
::メンバ名
```

この書き方は、自分で定義した名前空間に「グローバル名前空間と同じ名前のメンバがある」という状況で、グローバル名前空間のメンバを使いたい場合に役立ちます。

```cpp
#include <iostream>
#include <string>
using namespace std;

// グローバル名前空間のメンバとして MulAdd 関数を定義
int MulAdd(int a, int b, int c) {
  return a * b + c;
}

// グローバル名前空間のメンバとして Position 構造体を定義
struct Position {
  int x;
  int y;
};

namespace MyApp {
  // MyApp 名前空間のメンバとして Position 構造体を定義
  struct Position : public ::Position { // :: でグローバル名前空間の Position 構造体を指定
    int z;
  };

  // MyApp 名前空間のメンバとして MulAdd 関数を定義
  Position MulAdd(const Position& a, const Position& b, const Position& c) {
    Position d;
    d.x = ::MulAdd(a.x, b.x, c.x); // :: でグローバル名前空間の MulAdd 関数を指定
    d.y = ::MulAdd(a.y, b.y, c.y);
    d.z = ::MulAdd(a.z, b.z, c.z);
    return d;
}
```

とはいえ、「グローバル名前空間を指定する書き方」が役立つ場面は少ないです。C++の機能を深くまで使う必要にせまられない限り、知識として知っておく程度で十分でしょう。


### 3.5 無名名前空間

グローバル名前空間はアプリごとにひとつだけ存在し、ビルドに含まれるすべてのファイルで共通です。そのため、あるファイルのグローバル名前空間で変数を宣言すると、他のファイルでは同じ名前の変数を宣言できません。

もし同じ名前を付けてしまうと、「多重定義」というエラーになります。<br>
次のプログラムは、多重定義になる例です。

**コード**

```cpp
// A.cpp
#include <iostream>

int a = 10;


// B.cpp
#include <iostream>

int a = 5; // エラー. a という名前は A.cpp で宣言されている

int main() {
  std::cout << a << std::endl;
}
```

**実行結果**

```txt
usr/bin/ld: tmp/B.o:(.data+0x0): multiple definition of `a'; tmp/A.o:(.data+0x0): first defined here
collect2: error: ld returned 1 exit status
```

そのため、グローバル名前空間に変数や関数、構造体を定義するときは、毎回その他のファイルで同じ名前を使っていないことを確認しなくてはなりません。

この問題を解決するには、「無名名前空間(むめい・なまえくうかん)」を使います。

無名名前空間は、ファイルごとに独立した名前空間スコープを持ちます。これによって、他のファイルと名前が重複することを防げます。

「無名名前空間」を定義するには、次のように、名前を指定せずに`namespace`キーワードだけを使います。

```cpp
// A.cpp
namespace { // 無名名前空間の定義
  int a = 10;
} // 無名名前空間の終端


// B.cpp
namespace { // 無名名前空間
  int a = 5; // OK. B.cpp の無名名前空間は A.cpp の無名名前空間とは異なる
} // 無名名前空間の終端

int main() {
  std::cout << a << std::endl;
}
```

>ファイルの先頭付近に無名名前空間を定義して、ファイル内でしか使わない変数や関数、構造体は、常にその中に定義すると良い。


### 3.6 usingディレクティブ

名前空間は便利ですが、毎回「名前空間名::メンバ名」と書くのはわずらわしいものです。<br>
そんなときは`using`(ユージング)ディレクティブの出番です。<br>
`using`ディレクティブを使うと、そのスコープ内では「名前空間名::」の部分を省略できます。

`using`ディレクティブは次のように書きます。

```cpp
using namespace 名前空間名;
```

`namespace`キーワードは省略できません。`using`ディレクティブは、名前空間に対して作用するものだからです。

次のプログラムは、`using`ディレクティブを使った例です。

```cpp
#include <iostream>
using namespace std; // 実はこれも usingディレクティブ

namespace MyApp { // MyApp という名前の名前空間を定義

  struct Position { int x, y; };

  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }
}

int main() {
  using namespace MyApp; // MyApp を省略可能にする

  // 名前空間 MyApp のメンバを使う
  Position a = { 1, 2 };
  Position b = Add(a.x, a.y, 3, 4);
  cout << b.x << ' ' << b.y << endl;
}
```

スコープの中に書いた`using`ディレクティブは、スコープの外には影響しない点に注意してください。

例えば、上記のプログラムの場合、`MyApp::`を省略できるのは`main`関数の中だけです。もし`main`関数の下に他の関数を書いたとしても、その部分では`MyApp::`を省略できません。

ところで、上記のプログラムのコメントにも書いていますが、いつも書いている`using namespace std`も、`using`ディレクティブです。`using namespace std;`と書くことで、以後はファイルの終端まで`std::`を省略できるようになります。

>**【ヘッダファイルにusingディレクティブを書いてはいけない】**<br>
>ヘッダファイルのグローバル名前空間に、`using`ディレクティブを書いてはいけません。
>
>もし書いてしまうと、そのヘッダファイルをインクルードしたすべてのファイルで、名前空間を省略できてしまいます。便利そうに聞こえますが、実際にはむしろ不便になります。例えば、あるファイルが「その名前空間にある関数と同じ名前の関数」を書いていた場合、`using`ディレクティブによって突然エラーになってしまうのです。
>
>なお、構造体定義のスコープの中で`using`ディレクティブを書くことは問題ありません。


### 3.7 using宣言

「`using`宣言」を使うと、名前空間の特定のメンバだけ「名前空間名::」を省略できます。<br>
`using`宣言は次のように書きます。

```cpp
using 名前空間名::メンバ名;
```

複数のメンバについて省略可能にするには、それらを`,`(カンマ)で区切って指定します。

```cpp
using 名前空間名::メンバ名その１, 名前空間名::メンバ名その２, ...(必要なだけ続く);
```

次のプログラムは、`using`宣言を使った例です。

```cpp
#include <iostream>
using namespace std;

namespace MyApp {
  struct Position { int x, y; };

  Position Add(int ax, int ay, int bx, int by) {
    Position c = { ax + bx, ay + by };
    return c;
  }
}

int main() {
  using MyApp::Position; // using宣言

  // using宣言により Position の MyApp:: は省略できる
  Position a = { 1, 2 };
  Position b = MyApp::Add(a.x, a.y, 3, 4); // Add は using 宣言をしていないので MyApp:: を省略できない

  cout << b.x << ' ' << b.y << endl;
}
```

`using`宣言の動作は「指定された名前を現在のスコープに取り込む」というものです。





### 3.8 名前空間のネスト

名前空間の中に、さらに名前空間を定義できます。つまり、名前空間はネスト可能です。

それから、構造体やクラス、関数などと違って、名前空間は何度でも定義できます。<br>
コンピュータは、最初の定義を見つけると、新しい名前空間として定義します。2回目以降の定義を見つけると、同じ名前の名前空間にメンバを追加します。

次のプログラムは、名前空間のネストと複数回定義を使った例です。

**コード**

```cpp
#include <iostream>
using namespace std;

namespace MyApp {    // MyApp という名前の名前空間を定義
  namespace Sub {    // MyApp 名前空間の中に、 Sub という名前の名前空間を定義
    int number = 57; // MyApp::Sub 名前空間のメンバとして変数 number を宣言
  }                  // ここで Sub 名前空間の定義が終わる
}                    // ここで MyApp 名前空間の定義が終わる

namespace MyApp { // MyApp という名前の名前空間を定義
                  // (先に定義した同盟の名前空間と同じスコープを持つ)
  // MyApp 名前空間のメンバとして Item 構造体を宣言
  struct Item {
    string name;
    string text;
  };

  // MyApp 名前空間のメンバとして変数 item を宣言
  Item item = { "アプリ", "アプリのアイテム" };

} // ここで MyApp 名前空間の定義が終わる

int main() {
  // 名前空間 MyApp::Sub のメンバである number 変数を使う
  cout << MyApp::Sub::number << endl;

  // 名前空間 MyApp のメンバである item 変数を使う
  cout << MyApp::item.name << endl;

  // 名前空間 MyApp のメンバである Item 構造体を使う
  MyApp::Item item = { "メイン", "メイン関数のアイテム" };
  cout << item.name << endl; // この item は、 main 関数スコープのローカル変数(MyApp のメンバの item ではない)
}
```

**実行結果**

```txt
57
アプリ
メイン
```


### 3.9 名前空間エイリアス

ネストされた名前空間は、ある程度長い名前になりがちです。もちろん、`using`ディレクティブを使えば省略できます。しかし、完全に省略してしまうと名前の多重定義エラーが発生するなど、`using`ディレクティブを使えない場合もあります。

この問題は、「名前空間エイリアス」を使って「別名」を定義することで解決できます。<br>
名前空間エイリアスは次のように書きます。

```cpp
namespace 別名 = 名前空間名;
```

名前空間名エイリアスで定義した別名は、元の名前空間と同じものとして扱われます。

**コード**

```cpp
#include <iostream>
using namespace std;

namespace MyApp {
  namespace Sub {
    int number = 57;
  }
}

namespace MS = MyApp::Sub; // MS を MyApp::Sub の別名として定義

int main() {

  // MS は MyApp::Sub の別名なので、以下の2行は全く同じ意味になる
  cout << MS::number << endl; // 別名を使う
  cout << MyApp::Sub::number << endl; // 元の名前を使う
}
```

**実行結果**

```txt
57
57
```


#### 3.9.1 型エイリアス

`using`による別名定義は、名前空間だけでなく「型」に対しても使えます。これは「型(かた)エイリアス」と呼ばれます。

次のプログラムは、型エイリアスを使った例です。

```cpp
#include <iostream>
#include <vector>
using namespace std;

namespace MyApp {
  struct Item {
    string name;
    int price;
  };
}

int main() {
  using MyItem = MyApp::Item; // MyItem を MyApp::Item の別名として定義
  using ItemVec = vector<MyItem>; // ItemVec を vector<MyItem> の別名として定義

  ItemVec v(5); // 別名 ItemVec を使う
  for (int i = 0; i < (int)v.size(); i++) {
    cin >> v[i].name >> v[i].price;
  }

  for (int i = 0; i < (int)v.size(); i++) {
    const MyItem& e = v[i]; // 別名 MyItem を使う
    cout << i + 1 << ": " << e.name << " " << e.price << endl;
  }
}
```

名前空間エイリアスと型エイリアスの違いは、対象が「名前空間」か「型」かという点だけです。<br>
これらを区別する必要はないので、「`using`を使うと別名を定義できる」という認識で構いません。


### 名前空間の練習問題

&emsp;[問題７ マイ・アップ・ネームスペース](#scrollTo=B9NVw-IiG-Bx)<br>
&emsp;[問題８ 同じ名前だけど違うもの](#scrollTo=Xi7RnMP0HMPQ)<br>
&emsp;[問題９ ２つの円の交差](#scrollTo=Qr2OY-ypHPOX)


---

## <font face="BIZ UDゴシック">4 C++形式のキャスト</font>

---


### 4.1 C++で追加されたキャスト

`(型名)変数名`と書くことで、変数の値を「型名」に変換する「キャスト」ができます。<br>
この形式のキャストは、「C形式のキャスト」と呼ばれます。

次のプログラムは「C形式のキャスト」の例です。

```cpp
double a = 3.1415;
int b = (int)a; // キャスト
```

しかし、C形式のキャストは安全性に問題があります。例えば、次のように全く関係のない型を変換できてしまいます。

```cpp
#include <iostream>
using namespace std;

struct Apple { int i; };
struct Fighter { double i; };

int main(){
  Apple a = { 123 };
  Fighter* b = (Fighter*)&a; // AppleのアドレスをFighterのポインタにキャスト
  cout << b->i << endl; // NG. エラーは出ないが、どんな結果になるか分からない
}
```

このような、誤ってキャストしてしまう問題への対策として、C++では以下の4種類のキャスト式が追加されました。

* static_cast(スタティック・キャスト): 互換性のある型へのキャスト
* const_cast(コンスト・キャスト): `const`のない型へのキャスト
* dynamic_cast(ダイナミック・キャスト): 継承関係のある型へのキャスト
* reinterpret_cast(リ・インタープリト・キャスト): 非互換の型へのキャスト

キャストが複数あるのは、間違った使い方をしにくくするために、用途ごとに種類を分けているからです。



### 4.2 static_cast

`static_cast`(スタティック・キャスト)は「互換性のある型」へのキャストを行います。これには以下の種類があります。

* 大きい数値型から、小さい数値型に変換
* 整数型または列挙型から、別の列挙型に変換
* 基底クラスのポインタ型から、派生クラスのポインタ型に変換
* 基底クラスの参照型から、派生クラスの参照型に変換

スタティックキャストの書き方は次のとおりです。

```cpp
static_cast<変換先の型>(変換したい式);
```

次のプログラムは、`static_cast`を使った例です。

```cpp
// 大きい数値型(double)から小さい数値型(int)に変換
// ※キャストを書かないと警告が出力される
double a = 3.14;
int b = static_cast<int>(a);

// 列挙型(enum class)から整数型(int)に変換
enum class Fruit {
  apple,
  orange,
  banana,
};
int c = static_cast<int>(Fruit::orange); // c は 1 になる

// 基底クラス(Base)のポインタから派生クラス(Sub)のポインタに変換
struct Base {};
struct Sub : Base {};
Base* base = new Sub;
Sub* sub = static_cast<Sub*>(base);
Base* p = sub; // ※派生クラスから基底クラスへの変換にはキャスト不要
```

`static_cast`は次のようなキャストはできません。

* `const int`から`int`へのキャストはできない(`const_cast`を使う)
* 継承関係にない型へのキャストはできない(やってはいけない)
* 整数からポインタ型へのキャストはできない(`reinterpret_cast`を使う)

`enum class`から数値型への変換にはキャストが必要ですが、`enum`では不要です。理由は、C++はCとの互換性を維持するために、`enum`をCと同じ動作にしているからです。それに対して`enum class`はC++で追加されたもので、Cにはありません。そのため、より安全な設計が選ばれたのです。


### 4.3 const_cast

`const_cast`(コンスト・キャスト)を使うと、

&emsp;**constの付いたポインタを、constをはずしたポインタに変換**

できます。

しかし、`const`を付けるのは「値を変更できなくするため」です。そのため、`const_cast`を使って「変更できないはずの値」を変更しようとすると「未定義の動作」という論理エラーが発生します。

次のプログラムは、`const_cast`を使っても大丈夫な場合と危険な場合の例です。

```cpp
const int a = 10;  // const変数を宣言
const int* b = &a; // constポインタを宣言し、const変数を参照

int* c = const_cast<int*>(b); // OK. constポインタからconstを外す

int d = *c; // OK. 読み出すだけで変更はしていない
*c += 13;   // NG. const変数を変更しようとしている
```

そもそも、`const`を外したければ「変数または引数の宣言から`const`を消す」だけで済みます。<br>
ですから、`const_cast`が必要になることは滅多にありません。

それなのに`const_cast`という機能が存在するのは、「`const_cast`を使っても安全(かつ便利)な場合」があるからです。

「安全な場合」とは、「`const`の付いていない変数を参照している`const`付きポインタから、一時的に`const`を外す」場合です。

一般的な用途は、「`const`付きの引数を受け取る関数」の「`const`なしバージョン」を作成することです。<br>
次のプログラムは「2つの文字列を受け取って短いほうを返す」関数の例です。

```cpp
// 短い方の文字列を返す関数(const付きバージョン)
const char* shortestString(const char* a, const char* b) {
  const int la = strlen(a);
  const int lb = strlen(b);
  if (la < lb) {
    return la;
  }
  return lb;
}

// 短い方の文字列を返す関数(constなしバージョン)
char* shortestString(char* a, char* b) {
  // constを付けることで、const付きバージョンを呼び出す
  const char* c = shortestString(const_cast<const char*>(a), b);
  return const_cast<char*>(c); // constを外して返す
}
```

この例では、まず「`const`付きバージョンの`shortestString`関数」を作ります。`const`があることで「文字列を書き換えない」ことが保証されます。

もうひとつの「`const`なしバージョンの`shortestString`関数」では、`const_cast`によって引数`a`に`const`を付与することで、`const`付きバージョンを呼び出します。そうすることで、「文字列を書き換えない」ことが保証されます。

>2つの引数のどちらかが`const char*`なら「`const`付きバージョン」が選ばれます。このとき、もうひとつの`char*`型の引数は、自動的に`const char*`型に変換されます(制限の強い型への変換にはキャスト不要)。

そして最後に、戻り値`c`に付いている`const`を、`const_cast`で外します。変数`c`は引数`a`か引数`b`のどちらかのコピーであり、いずれも元は`char*`型です。`const`を外しても「元の型に戻るだけ」なので安全です。

この書き方には、「書き換えない保証」だけではなく、「同じコードを2回書く手間がない」という利点もあります。

>`const_cast`は極力使用しないほうがよい。<br>
>それでも、もし使う場合は、参照している変数が絶対に`const`にならないと保証すること。


### 4.4 dynamic_cast

`dynamic_cast`(ダイナミック・キャスト)は、

&emsp;**基底クラスのポインタを、派生クラスのポインタに「安全に」変換**

します。

`static_cast`でも、基底クラスのポインタを派生クラスのポインタに変換できます。しかし、もしも「変換先のクラスと実際のクラスが異なる」場合は、実行時エラーや論理エラーになる危険があります。

次のプログラムは、`static_cast`による **危険なキャスト** の例です。

```cpp
struct Base {};
struct X : Base {
  int a = 1;
};
struct Y : Base {
  int a = 2, b = 3;
};

Base* x = new X;
Y* y = static_cast<Y*>(x); // NG. X型のオブジェクトをY型に変換している(ただし、この時点ではエラーにはならない)
cout << y->b << endl; // NG. 実行時エラー、または論理エラーが起きる可能性がある
```

これに対して`dynamic_cast`では、「ポインタが指すオブジェクトの型が派生クラスであることを検証する」という処理が行われます。そして、派生クラスではなかった場合、実行時エラーや論理エラーの代わりに`nullptr`が返される」のです。

エラーになる前に型をチェックできるので「安全に変換」できるというわけです。

次のプログラムは、`dynamic_cast`による **安全なキャスト** の例です。

```cpp
struct Base {};
struct X : Base {
  int a = 1;
};
struct Y : Base {
  int a = 2, b = 3;
};

Base* x = new X;
Y* y = dynamic_cast<Y*>(x); // OK. x は Y型ではないので nullptr が返される
if (y) { // y が nullptr 以外の場合に実行
  cout << y->b << endl; // OK. nullptr ではなかったのでY型で間違いない
}
```

このように、`dynamic_cast`の特徴は

&emsp;**実際に参照している派生クラスを検査できる**

ということです。

>**【dynamic_castより仮想関数を使おう】**<br>
>継承を使う主な理由は「複数のクラスを共通化して扱いたい」からです。そして、共通化の主役は仮想関数です。<br>
>まず「仮想関数で実現できないか？」を考え、`dynamic_cast`の使用は「他に方法が思いつかない」場合に限りましょう。


#### 4.4.1 dynamic_castで注意すること

`dynamic_cast`を使うと、他のキャストより処理が遅くなる場合があります。その理由は、

&emsp;**dynamic_castは、他のキャストより「メモリを消費」したり「時間がかかる」場合がある**

からです。

特に、派生クラスの種類が多かったり、継承関係が複雑になるほどメモリも時間も必要となります。<br>
そのため、参照している派生クラスが確実に分かっている場合は、`static_cast`を使うべきです。

>**【dynamic_castとstatic_castの使い分け】**<br>
>`dynamic_cast`は、基底クラスのポインタが、どの派生クラスを参照しているか分からない場合に使う。<br>
>派生クラスが確実に分かる場合は`static_cast`を使う。

もう一つの注意点として、

&emsp;**コンストラクタやデストラクタでは、thisをdynamic_castしない**

というものがあります。これは、次の理由によります。

* コンストラクタではまだ派生クラスが初期化されていない
* デストラクタでは派生クラスは既に解体されている

いずれの場合も派生クラスはまだ存在していないか、既に存在していないので、そのメンバは使えないのです。


#### 4.4.2 RTTI(実行時型情報)

`dynamic_cast`を使うには、プロジェクトの設定で RTTI(アール・ティー・ティー・アイ)が有効になっている必要があります。

RTTIは`Run-Time Type Information`(ランタイム・タイプ・インフォメーション)の頭文字で、日本語では「実行時型情報(じっこうじ かた じょうほう)」といいます。名前のとおり、すべての型の情報をデータ化したものです。RTTIが有効な場合、実行時型情報はプログラムに自動的に埋め込まれます。

RTTIが埋め込まれると、アプリの実行に必要なメモリが多少増えます。また、`dynamic_cast`を使うこと自体が実行速度を低下させる場合もあります。

これらの理由から、プロジェクトによってはRTTIを無効にしていることがあります。無効になっている場合は`dynamic_cast`は事実上使えません(書くことはできるがほぼ実行時エラーになる)。


### 4.5 reinterpret_cast

`reinterpret_cast`(リインタープリト・キャスト)を説明する前に、警告しておきます。

&emsp;**reinterpret_castは極めて危険です。使わないでください。**<br>
&emsp;**reinterpret_castは極めて危険です。使わないでください。**<br>
&emsp;**reinterpret_castは極めて危険です。使わないでください。**<br>

三回言いましたよ。使わないでくださいね。

それでも、どうしても必要な場合もあります(だから、この機能が存在するのです)。<br>
そこで、以下では使い方と危険性を説明します。

さて、`reinterpret_cast`(リ・インタープリト・キャスト)は、「互換性の **ない** 型」へのキャストを行います。これには以下の種類があります。

* ある型のポインタを、別の型のポインタに変換
* ある型の参照を、別の型の参照に変換
* ある型のポインタを、十分な大きさを持つ整数型に変換
* 整数型を、ある型のポインタに変換


#### 4.5.1 ポインタ(参照)同士の変換

最初の2つの種類の変換は、「ポインタ同士の変換」です。参照も内部的にはポインタなので、ここに含まれます。

まず、元のポインタ型に戻すことを前提とする限り、`reinterpret_cast`は基本的にどんなポインタ型への変換も可能です。ただし、変換後の型を使った操作は、「相互変換可能なポインタ型である」という条件を満たす場合のみ許可されます。

「相互変換可能なポインタ」というのは、参照しているアドレスに「同等のオブジェクトが存在する」ことです。例えば、次のプログラムでは、構造体変数`t`と、そのメンバ変数`t.s`は同じアドレスに存在します。

```cpp
struct S {
  int a;
  int b;
};

struct T {
  S s;
  int x;
  int y;
  double z;
};
T t;
S* p = reinterpret_cast<S*>(&t); // OK. T 型の先頭メンバは S 型なので変換可能
T* q = reinterpret_cast<T*>(p);  // OK. p は T 型の先頭メンバを指している

S* a = &t.s; // OK. S 型の変数を指すポインタ
T* b = reinterpret_cast<T*>(a); // OK. a は T 型の先頭メンバを指している

int* u = &t.x; // OK. int型の変数を指すポインタ
//T* v = reinterpret_cast<T*>(u); // NG. u は T 型の先頭メンバではない
```

C++の仕様では、`T`型や`S`型が仮想関数を持たない限り、上記の変数`t`のアドレスと`t.s`のアドレスは一致します。これが「相互変換可能」の意味です。

>他にもいろいろ条件があるのですが、`reinterpert_cast`なんて使わないに越したことはないので、細かい条件は覚えなくていいです。どうしても必要になったときに調べましょう。

この仕様は、C言語で継承のような仕組みを実現するために使われています。もちろん、C++言語で「`T`を`S`の一種として扱いたい」場合は素直に継承を使うべきです。

このように、C言語のプログラムをC++から利用するとき、`reinterpret_cast`を必要とする場合があります。

>**【相互変換可能かどうかはプログラマが保証する】**<br>
>`reinterpret_cast`は相互変換可能なポインタかどうかを検査しません。<br>
>「相互変換可能なポインタである」ということは、プログラマが保証しなくてはなりません。もし「相互変換できないポインタ」に変換しようとすると、実行時エラーや論理エラーが発生します。


#### 4.5.2 整数型とポインタ型の変換

残りの2つの変換は、整数型とポインタ型を相互に変換するものです。<br>
これにも条件があり、「ポインタ型のビット表現を完全に格納可能なサイズを持つ整数型であること」が条件となります。

「ビット表現を完全に格納可能」というのは、つまり

&emsp;**ポインタ型のビット数 ≧ 整数型のビット数**

でなくてはならない、ということです。

ポインタ型のビット数は、64bitシステムの場合は64ビット、32bitシステムの場合は32ビットです。そのため、64bitシステムでは`long long`以上、32bitシステムでは`int`以上のサイズを持つ整数型が要求されます。

>なお、 Google Colab は64bitシステムになっています。

ただし、C++の整数型はシステムによって変わる可能性があります。そのため、ヘッダファイル`stdint.h`において、`intptr_t`(イント・ポインタ・ティー)という「ポインタ型のサイズ以上だと保証された整数型」が定義されています。

もし、ポインタ型と整数型の変換が必要になった場合は、この`intptr_t`型を使うとよいでしょう。次のプログラムは、ポインタ型と整数型を互いに変換する例です。

**コード**

```cpp
#include <iostream>
#include <iomanip>
#include <stdint.h>
using namespace std;

struct S {
  int a = 1;
  double b = 2;
};

int main() {
  S s;
  intptr_t i = reinterpret_cast<intptr_t>(&s); // s のアドレスを整数型に変換
  cout << hex << i << endl; // 変換した整数を16進数で出力
  cout << &s << endl;       // s のアドレスを出力

  S* p = reinterpret_cast<S*>(i); // i を S のポインタに変換
  cout << p->a << ',' << p->b << endl;
}
```

**実行結果**

```txt
7ffd3c029b20
0x7ffd3c029b20
1,2
```

また、メモリを節約するためや利便性のために、ポインタと数値のどちらかを格納可能な変数を定義することもあります。次のプログラムは、ポインタ型の値を整数として使う例です。

**コード**

```cpp
#include <iostream>
#include <stdint.h>
using namespace std;

// modeの値によってdataの扱いが変わる関数
intptr_t func(int mode, int* data) {
  if (mode == 0) {
    return reinterpret_cast<intptr_t>(data); // 整数として出力
  }
  return *data; // ポインタの参照先を出力
}

int main() {
  cout << func(0, reinterpret_cast<int*>(123)) << endl;
  //cout << func(1, reinterpert_cast<int*>(123)) << endl; // NG. mode の番号を間違えている

  int i = 456;
  cout << func(1, &i) << endl;
  //cout << func(0, &i) << endl; // NG. mode の番号を間違えている
}
```

**実行結果**

```txt
123
456
```

当然ですが、関数`func`の使い方を少しでも間違えると実行時エラーや論理エラーが発生します。

>**【reinterpret_castは使わないこと】**<br>
>`reinterpret_cast`は非常に危険です。<br>
>C言語のプログラムと連携する場合など、他に方法がない場合を除いて、`reinterpret_cast`を使わないこと。


### キャストの練習問題

&emsp;[問題１０ 型が違います](#scrollTo=KMPSIU5YHzWv)<br>
&emsp;[問題１１ 人間と幽霊と木](#scrollTo=tTl0xTyLH9zZ)<br>
&emsp;[問題１２ 数値か、文章か](#scrollTo=-2c7gR_UH_PL)


----

## 5 練習問題

----

以下の手順にしたがって、各問題のプログラムを完成させなさい。

1. `%%writefile ...`の下の行からがプログラムです。問題文に従ってプログラムを修正、または追加してください。
2. プログラムを修正したら、セルの左側にある`▶`をクリックします。すると、ファイルが保存されます。
3. 「動作テスト」セルの`▶`をクリックすると、2で保存したファイルがコンパイル＆実行され、実行結果が表示されます。<br>
   このセルは、修正したプログラムの動作を確認するために使ってください。
4. 「実行」セルの`▶`をクリックすると、2で保存したファイルがコンパイル＆実行され、結果の成否が判定されます。
5. 判定に成功したら`AC`と表示されます。次の問題に進んでください。
6. 失敗したら`WA`と表示されます(その前にエラーメッセージが表示される場合もあります)。<br>
   これは、プログラムのどこかにエラーがあることを意味します。<br>
   「動作テスト」を使ってエラーを修正し、`AC`を目指してください。


### ❓️問題１ 配列を取り込む

`int.txt`と`str.txt`という2つのファイルがあります。<br>
それぞれのファイルの内容は次のとおりです。

**int.txt**<br>
&emsp;3,1,4,1,5,9,2,6,5,3,5

**str.txt**<br>
&emsp;"alpha","bravo","charlie","delta"

この2つのファイルを配列としてインクルードし、出力するプログラムを作っています。<br>
プログラム中の２つの`？`記号を適切なプリプロセッサ指令で書き換えて、プログラムを完成させなさい。


**出力例**

```txt
3 1 4 1 5 9 2 6 5 3 5
alpha bravo charlie delta
```



In [ ]:
%%writefile practice_01a.cpp
#include <iostream>
#include <vector>
using namespace std;

int main() {

  vector<int> vint = {
    ？
  };
  for (int i = 0; i < (int)vint.size(); i++) {
    cout << vint[i] << ' ';
  }
  cout << endl;

  vector<string> vstr = {
    ？
  };
  for (int i = 0; i < (int)vstr.size(); i++) {
    cout << vstr[i] << ' ';
  }
  cout << endl;
}

In [ ]:
# @title 動作テスト
!echo "3,1,4,1,5,9,2,6,5,3,5" > int.txt
!echo '"alpha","bravo","charlie","delta"' > str.txt
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_01a practice_01a.cpp && ./practice_01a

In [ ]:
# @title 実行
!echo "3,1,4,1,5,9,2,6,5,3,5" > int.txt
!echo '"alpha","bravo","charlie","delta"' > str.txt
!diff -Z <(echo -e "3 1 4 1 5 9 2 6 5 3 5\nalpha bravo charlie delta") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_01a practice_01a.cpp && ./practice_01a) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_01a.cpp
#include <iostream>
#include <vector>
using namespace std;

int main() {

  vector<int> vint = {
#include "int.txt"
  };
  for (int i = 0; i < (int)vint.size(); i++) {
    cout << vint[i] << ' ';
  }
  cout << endl;

  vector<string> vstr = {
#include "str.txt"
  };
  for (int i = 0; i < (int)vstr.size(); i++) {
    cout << vstr[i] << ' ';
  }
  cout << endl;
}

### ❓️問題２ 角度の単位

円周の角度を 360° であらわす方法を「度数法(どすうほう)」といいます。単位は「度」または「ディグリー」です。<br>
また、円周の角度を $ 2 \pi $ であらわす方法を「弧度法(こどほう)」といいます。単位は「ラジアン」です。

度数法の数値を弧度法に変換したり、逆に、弧度法の数値を度数法に変換するプログラムを作りたいです。<br>
度数法の数値を $ D $ 、弧度法の数値を $ R $ 、円周率を $ \pi $ とすると、それぞれの変換は次の式で行えます。

&emsp;$ R = D \div 180 \times \pi $

&emsp;$ D = R \div \pi \times 180 $

種類をあらわす文字と、角度をあらわす数値の2つが入力されます。<br>
文字は、度数法の場合は文字`d`、弧度法の場合は文字`r`です。<br>
円周率のマクロ`PI`を使って、度数法と弧度法を相互に変換するプログラムを完成させなさい。

**入力データ例（１）**

```txt
d 45
```

**出力例（１）**

```txt
0.785398
```

**入力データ例（２）**

```txt
r 1.6
```

**出力例（２）**

```txt
91.6732
```


In [ ]:
%%writefile practice_01b.cpp
#include <iostream>
using namespace std;

#define PI 3.1415926535

int main() {

  // この下に、度数法と弧度法を、相互に変換して出力するプログラムを書く
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_01b practice_01b.cpp && echo "この下をクリックして、種類と角度を入力" && ./practice_01b

In [ ]:
# @title 実行
!diff -Z <(echo -e "0.785398\n91.6732\n0\n0\n180") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_01b practice_01b.cpp && echo "d 45" | ./practice_01b && echo "r 1.6" | ./practice_01b && echo "d 0" | ./practice_01b && echo "r 0" | ./practice_01b && echo "r 3.1415925635" | ./practice_01b) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_01b.cpp
#include <iostream>
using namespace std;

#define PI 3.1415926535

int main() {

  // この下に、度数法と弧度法を、相互に変換して出力するプログラムを書く
  char a;
  double b;
  cin >> a >> b;

  if (a == 'd') {
    cout << b / 180 * PI << endl;
  } else {
    cout << b / PI * 180 << endl;
  }
}


### ❓️問題３ 繰り返しマクロ

繰り返しマクロ`REP`(レプ)と、データ出力マクロ`OUTPUT`(アウトプット)を使って、変数`i`を`0`から`9`まで出力するプログラムを作っています。

出力例と同じ出力になるように、プログラム中の2ヶ所の「？」を適切なプログラムに書き換えなさい。

**出力例**

```txt
i=0
i=1
i=2
i=3
i=4
i=5
i=6
i=7
i=8
i=9
```


In [ ]:
%%writefile practice_01c.cpp
#include <iostream>
using namespace std;

#define REP(i, n) for (int i = 0; i < n; i++)

#define OUTPUT(x) cout << #x << "=" << x << endl;

int main() {
  // ここから上は変更しない

  // 以下の2つの ？ を、出力邸と同じになるように書き換える
  REP( ？ ) {
    OUTPUT( ？ )
  }
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_01c practice_01c.cpp && ./practice_01c

In [ ]:
# @title 実行
!diff -Z <(echo -e "i=0\ni=1\ni=2\ni=3\ni=4\ni=5\ni=6\ni=7\ni=8\ni=9") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_01c practice_01c.cpp && ./practice_01c) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_01c.cpp
#include <iostream>
using namespace std;

#define REP(i, n) for (int i = 0; i < n; i++)

#define OUTPUT(x) cout << #x << "=" << x << endl;

int main() {
  // ここから上は変更しない

  // 以下の2つの ？ を、出力邸と同じになるように書き換える
  REP(i, 10) {
    OUTPUT(i)
  }
}

### ❓️問題４ バージョン2のアプリ

あるアプリは何度かバージョンアップが行われ、バージョンごとに次の機能が使えます。

* バージョン1: 機能A
* バージョン2: 機能A, 機能B
* バージョン3: 機能B, 機能C

求めるバージョンのアプリをビルドするには、マクロ`APP_VER`を定義し、バージョン番号を指定します。

バージョン2のアプリがビルドされるように、適切なマクロ定義を追加しなさい。

**出力例**

```txt
バージョン:2
利用可能: 機能 A、機能 B
```


In [ ]:
%%writefile practice_01d.cpp
#include <iostream>
using namespace std;

// この下にマクロを定義する


// ここから下は変更しない
int main()
{
  cout << "バージョン:" << APP_VER << endl;

#if APP_VER >= 3
  cout << "利用可能: 機能 B、機能 C" << endl;
#elif APP_VER >= 2
  cout << "利用可能: 機能 A、機能 B" << endl;
#else
  cout << "利用可能: 機能 A" << endl;
#endif
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_01d practice_01d.cpp && ./practice_01d

In [ ]:
# @title 実行
!diff -Z <(echo -e "バージョン:2\n利用可能: 機能 A、機能 B") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_01d practice_01d.cpp && ./practice_01d) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_01d.cpp
#include <iostream>
using namespace std;

// この下にマクロを定義する
#define APP_VER 2

// ここから下は変更しない
int main()
{
  cout << "バージョン:" << APP_VER << endl;

#if APP_VER >= 3
  cout << "利用可能: 機能 B、機能 C" << endl;
#elif APP_VER >= 2
  cout << "利用可能: 機能 A、機能 B" << endl;
#else
  cout << "利用可能: 機能 A" << endl;
#endif
}


### ❓️問題５ 移動攻撃

`switch`文を使って、防御、移動、攻撃、移動攻撃、の4種類の行動を選択するプログラムがあります。<br>
「移動攻撃」は「移動してから攻撃」なので、「移動」の`case`文から`break`せずに「攻撃」の`case`文を実行するようにしています。

プログラム自体は意図したとおりに機能しましたが、コンパイラの警告が表示されます。警告が表示されないように、適切な場所に属性を追加しなさい。

**入力する番号と行動の対応表**

| 番号 | 行動 |
|:----:|:----:|
|   0  | 防御 |
|   1  | 移動 |
|   2  | 攻撃 |
|   3  | 移動攻撃 |

**出力例**

```txt
3
移動攻撃
```


In [ ]:
%%writefile practice_02a.cpp
#include <iostream>
using namespace std;

int main() {
  int a;
  cin >> a;

  switch (a) {
  case 0:
    cout << "防御";
    break;

  case 1:

  case 3:
    cout << "移動";
    if (a == 1) {
      break;
    }

  case 2:
    cout << "攻撃";
    break;
  }

  cout << endl;
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_02a practice_02a.cpp && echo "この下をクリックして、行動の番号を入力" && ./practice_02a

In [ ]:
# @title 実行
!diff -Z <(echo -e "防御\n移動\n攻撃\n移動攻撃\n") <(g++ -std=c++20 -O2 -Wall -Wextra -Werror -o practice_02a practice_02a.cpp && echo 0 | ./practice_02a && echo 1 | ./practice_02a && echo 2 | ./practice_02a && echo 3 | ./practice_02a && echo 4 | ./practice_02a) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_02a.cpp
#include <iostream>
using namespace std;

int main() {
  int a;
  cin >> a;

  switch (a) {
  case 0:
    cout << "防御";
    break;

  case 1:

  case 3:
    cout << "移動";
    if (a == 1) {
      break;
    }
    [[fallthrough]];

  case 2:
    cout << "攻撃";
    break;
  }

  cout << endl;
}


### ❓️問題６ 使われないかもしれない

4つの引数`a`, `b`, `c`, `d`を受け取って、計算結果を返す`MultiplyAddDiv`(マルチプライ・アド・ディブ)という関数があります。<br>
この関数を使うアプリの開発中にさまざまなことがあり、最終的に引数`d`は使われなくなりました。

理想的なのは、「引数を3つに減らして、すべての関数呼び出しから引数`d`をなくす」ことでしょう。ですが、この関数は既にあちこちで使われていたので、「引数を3つに減らして、すべての呼び出しから引数`d`をなくす」のは現実的ではないと判断されました。

残る問題は、「引数`d`が使われていません」という警告をなくすことだけです。<br>
引数`d`に適切な属性を追加して、警告をなくしてください。

**入力例**

```txt
2 3 5
```

**出力例**<br>
&emsp;11

In [ ]:
%%writefile practice_02b.cpp
#include <iostream>
using namespace std;

// a * b + c を計算する関数
// 補足: 以前は (a * b + c) / d という式だった。
//       調査の結果、ほとんどの呼び出しで d には 1.0 が指定されていた。
//       計算を速くするために、式から d を除外した。
int MultiplyAddDiv(double a, double b, double c, double d) {
  //return (a * b + c) / d;
  return a * b + c;
}

int main() {
  int a, b, c;
  cin >> a >> b >> c;
  cout << MultiplyAddDiv(a, b, c, 1.0) << endl;
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_02b practice_02b.cpp && echo "この下をクリックして、3つの数字を入力" && ./practice_02b

In [ ]:
# @title 実行
!diff -Z <(echo -e "11\n4132") <(g++ -std=c++20 -O2 -Wall -Wextra -Werror -o practice_02b practice_02b.cpp && echo "2 3 5" | ./practice_02b && echo "11 12 4000" | ./practice_02b) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_02b.cpp
#include <iostream>
using namespace std;

// a * b + c を計算する関数
// 補足: 以前は (a * b + c) / d という式だった。
//       調査の結果、ほとんどの呼び出しで d には 1.0 が指定されていた。
//       計算を速くするために、式から d を除外した。
int MultiplyAddDiv(double a, double b, double c, [[maybe_unused]] double d) {
  //return (a * b + c) / d;
  return a * b + c;
}

int main() {
  int a, b, c;
  cin >> a >> b >> c;
  cout << MultiplyAddDiv(a, b, c, 1.0) << endl;
}


### ❓️問題７ マイ・アップ・ネームスペース

`MyApp`という名前空間を定義し、その中に以下のメンバ定義を追加してください。

**Position構造体**

* 構造体名: `Position`(ポジション)
* メンバ変数1: `int`型で名前は`x`
* メンバ変数2: `int`型で名前は`y`

**Add関数**

* 関数名: `Add`(アド)
* 第１引数: 点AのX座標
* 第２引数: 点AのY座標
* 第３引数: 点BのX座標
* 第４引数: 点BのY座標
* 戻り値: 点AとBの座標を足した結果を`Position`型で返す
* 機能:
  1. `Position`型の変数`c`を宣言
  2. `c`のメンバ変数`x`に「点AのX座標 + 点BのX座標」を代入
  3. `c`のメンバ変数`y`に「点AのY座標 + 点BのY座標」を代入
  4. `c`を返す

**入力データ例**

```txt
1 2
3 4
```

**出力例**

```txt
4 6
```


In [ ]:
%%writefile practice_03a.cpp
#include <iostream>
using namespace std;

// この下に、名前空間 MyApp を定義し、メンバを書く


// ここから下は変更しない
int main() {

  int ax, ay, bx, by;
  cin >> ax >> ay >> bx >> by;

  MyApp::Position c = MyApp::Add(ax, ay, bx, by);
  cout << c.x << ' ' << c.y << endl;
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_03a practice_03a.cpp && echo "この下をクリックして、点AのX, Y座標と、点BのX, Y座標を入力" && ./practice_03a

In [ ]:
# @title 実行
!diff -Z <(echo -e "4 6\n107 94\n-5 -11") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_03a practice_03a.cpp && echo "1 2 3 4" | ./practice_03a && echo "73 16 34 78" | ./practice_03a && echo "-50 13 45 -24" | ./practice_03a) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_03a.cpp
#include <iostream>
using namespace std;

// この下に、名前空間 MyApp を定義し、メンバを書く
namespace MyApp {

  struct Position {
    int x;
    int y;
  };

  Position Add(int ax, int ay, int bx, int by) {
    Position c;
    c.x = ax + bx;
    c.y = ay + by;
    return c;
  }
}

// ここから下は変更しない
int main() {

  int ax, ay, bx, by;
  cin >> ax >> ay >> bx >> by;

  MyApp::Position c = MyApp::Add(ax, ay, bx, by);
  cout << c.x << ' ' << c.y << endl;
}


### ❓️問題８ 同じ名前だけど違うもの

グローバル名前空間に`MultiplyAdd`(マルチプライ・アド)という名前の、3つの数 A, B, C が与えられると $ A \times B + C $ を計算して返す関数があります。

この関数を利用して、とあるアプリ専用の二次元座標A, B, Cについて、 $ A \times B + C $ を計算する、新しい関数を作りたいです。<br>
処理の構造は同じなので、新しい関数の名前も`MultiplyAdd`にしました。

ところが、新しい`MultiplyAdd`関数の中で、グローバルな`MultiplyAdd`関数を呼び出そうとすると、エラーになってしまいます。<br>
グローバルな`MutiplyAdd`関数を使いながら、エラーを出さずに計算が行われるように、プログラムを修正しなさい。

**入力データ例**

```txt
1 2 3 4 5 6
```

**出力例**

```txt
8 14
```


In [ ]:
%%writefile practice_03b.cpp
#include <iostream>
using namespace std;

// a と b を掛けて c を足す関数
int MultiplyAdd(int a, int b, int c) {
  return a * b + c;
}

namespace MyApp {

  struct Position { int x, y; };

  // a と b を掛けて c を足す関数(Position構造体バージョン)
  Position MultiplyAdd(const Position& a, const Position& b, const Position& c) {
    Position d;
    // ここから上は変更しない

    // この下の2行がエラーになる
    // グローバルな MultiplyAdd 関数を呼び出してもエラーが出ないように修正する
    d.x = MultiplyAdd(a.x, b.x, c.x);
    d.y = MultiplyAdd(a.y, b.y, c.y);

    // ここから下は変更しない
    return d;
  }
}

int main() {
  MyApp::Position a, b, c;
  cin >> a.x >> a.y >> b.x >> b.y >> c.x >> c.y;

  MyApp::Position d = MyApp::MultiplyAdd(a, b, c);
  cout << d.x << ' ' << d.y << endl;
}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_03b practice_03b.cpp && echo "この下をクリックして、3つの点A, B, CのX, Y座標を入力" && ./practice_03b

In [ ]:
# @title 実行
!diff -Z <(echo -e "8 14\n78 84") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_03b practice_03b.cpp && echo "1 2 3 4 5 6" | ./practice_03b && echo "11 52 8 2 -10 -20" | ./practice_03b) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_03b.cpp
#include <iostream>
using namespace std;

// a と b を掛けて c を足す関数
int MultiplyAdd(int a, int b, int c) {
  return a * b + c;
}

namespace MyApp {

  struct Position { int x, y; };

  // a と b を掛けて c を足す関数(Position構造体バージョン)
  Position MultiplyAdd(const Position& a, const Position& b, const Position& c) {
    Position d;
    // ここから上は変更しない

    // この下の2行がエラーになる
    // グローバルな MultiplyAdd 関数を呼び出してもエラーが出ないように修正する
    d.x = ::MultiplyAdd(a.x, b.x, c.x); // :: を追加
    d.y = ::MultiplyAdd(a.y, b.y, c.y); // :: を追加

    // ここから下は変更しない
    return d;
  }
}

int main() {
  MyApp::Position a, b, c;
  cin >> a.x >> a.y >> b.x >> b.y >> c.x >> c.y;

  MyApp::Position d = MyApp::MultiplyAdd(a, b, c);
  cout << d.x << ' ' << d.y << endl;
}


### ❓️問題９ ２つの円の交差

円の交差判定を行うプログラムを作っています。<br>
`main`関数の中に、次の処理をする動作確認プログラムを作成してください。<br>
交差判定に使うプログラムは名前空間`MyApp::Engine::Collision`に書かれています。

なお、名前空間名が長すぎるので、名前空間エイリアスを使って別名`MEC`を定義しています。<br>
作成するプログラムでは、必ず`MEC`を使ってください。

**作成するプログラムの概要**

1. 2つの円をあらわす変数`a`と`b`を宣言する
2. 標準入力から`a`、`b`の順で、円のデータを読み込む
3. `Overlap`(オーバーラップ)関数を使って、`a`と`b`が交差しているかどうかを調べる
4. `a`と`b`が交差している場合、標準出力に文字列`"hit"`と改行を出力する
5. `a`と`b`が交差していない場合、標準出力に文字列`"miss"`と改行を出力する

**入力データ例（１）**

```txt
1 2 1 4 3 2
```

**出力例（１）**

```txt
miss
```

**入力データ例（２）**

```txt
-5 3 6 8 -2 8
```

**出力例（２）**

```txt
hit
```


In [ ]:
%%writefile practice_03c.cpp
#include <iostream>
using namespace std;

namespace MyApp {
  namespace Engine {
    namespace Collision {

      // 円クラス
      struct Circle {
        double x, y;   // X座標とY座標
        double radius; // 半径
      };

      // 2つの円の交差を判定する関数
      // 戻り値が true  の場合: 交差している
      //          false の場合: 交差していない
      bool Overlap(const Circle& a, const Circle& b) {
        double dx = a.x - b.x;
        double dy = a.y - b.y;
        double rr = a.radius + b.radius;
        return dx * dx + dy * dy <= rr * rr;
      }

    } // namespace Collision
  } // namespace Engine
} // namespace MyApp

// 別名を定義
namespace MEC = MyApp::Engine::Collision;

#define MyApp
#define Engine
#define Collision

int main() {
  // ここから上は変更しない

  // この下に、2つの円のデータを入力させて、交差判定を行うプログラムを書く

}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_03c practice_03c.cpp && echo "この下をクリックして、円AのXY座標と半径、円BのXY座標と半径を入力" && ./practice_03c

In [ ]:
# @title 実行
!diff -Z <(echo -e "hit\nmiss\nhit\nmiss") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_03c practice_03c.cpp && echo "-1 -1 2 2 0 2" | ./practice_03c && echo "5 7 1 2 1 1" | ./practice_03c && echo "2 2 2 4 2 2" | ./practice_03c && echo "15 8 9 -2 -3 11" | ./practice_03c) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_03c.cpp
#include <iostream>
using namespace std;

namespace MyApp {
  namespace Engine {
    namespace Collision {

      // 円クラス
      struct Circle {
        double x, y;   // X座標とY座標
        double radius; // 半径
      };

      // 2つの円の交差を判定する関数
      // 戻り値が true  の場合: 交差している
      //          false の場合: 交差していない
      bool Overlap(const Circle& a, const Circle& b) {
        double dx = a.x - b.x;
        double dy = a.y - b.y;
        double rr = a.radius + b.radius;
        return dx * dx + dy * dy <= rr * rr;
      }

    } // namespace Collision
  } // namespace Engine
} // namespace MyApp

// 別名を定義
namespace MEC = MyApp::Engine::Collision;

#define MyApp
#define Engine
#define Collision

int main() {
  // ここから上は変更しない

  // この下に、2つの円のデータを入力させて、交差判定を行うプログラムを書く
  MEC::Circle a;
  MEC::Circle b;
  cin >> a.x >> a.y >> a.radius;
  cin >> b.x >> b.y >> b.radius;

  if (MEC::Overlap(a, b)) {
    cout << "hit" << endl;
  } else {
    cout << "miss" << endl;
  }
}


### ❓️問題１０ 型が違います

キャストさんは、「さまざまな型の変数を、異なる型の変数に代入する実験」をしています。ですが、「できる」と聞いていたいくつかの代入がうまくいかないことに気が付きました。<br>
調査の結果、代入を成功させるには、`static_cast`という機能が必要なことを突き止めました。しかし、結局キャストさんには使いかたが分からず、エラーを直せませんでした。

キャストさんの代わりに、以下のプログラムの適切な箇所に`static_cast`を追加して、プログラムを実行できるようにしてください。

**出力例**

```txt
3
1
1 2
```


In [ ]:
%%writefile practice_04a.cpp
#include <iostream>
using namespace std;

// 果物の列挙型
enum class Fruit { apple, orange, banana };

// 基底クラス
struct Base {
  virtual ~Base() = default;

  int b = 1;
};

// 派生クラス
struct Sub : Base {
  virtual ~Sub() = default;

  int s = 2;
};

int main() {

  // 大きい数値型(double)から小さい数値型(int)に変換
  double a = 3.14;
  int b = a;
  cout << b << endl;

  // 列挙型(enum)から整数型(int)に変換
  int c = Fruit::orange;
  cout << c << endl;

  // 基底クラス(Base)のポインタから派生クラス(Sub)のポインタに変換
  Base* base = new Sub;
  Sub* sub = base;
  cout << sub->b << ' ' << sub->s << endl;
  delete base;
}


In [ ]:
# @title 動作テスト
!cat <(echo) practice_04a.cpp > practice_04a_.cpp
!g++ -std=c++20 -O2 -Wall -Wextra -Wconversion -o practice_04a practice_04a_.cpp && ./practice_04a
!rm practice_04a_.cpp

In [ ]:
# @title 実行
!diff -Z <(echo -e "3\n1\n1 2") <(g++ -std=c++20 -O2 -Wall -Wextra -Wconversion -Werror -o practice_04a practice_04a.cpp && ./practice_04a) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_04a.cpp
#include <iostream>
using namespace std;

// 果物の列挙型
enum Fruit { apple, orange, banana };

// 基底クラス
struct Base {
  virtual ~Base() = default;

  int b = 1;
};

// 派生クラス
struct Sub : Base {
  virtual ~Sub() = default;

  int s = 2;
};

int main() {

  // 大きい数値型(double)から小さい数値型(int)に変換
  double a = 3.14;
  int b = static_cast<int>(a); // ここに static_cast を追加
  cout << b << endl;

  // 列挙型(enum)から整数型(int)に変換
  int c = static_cast<int>(Fruit::orange); // ここに static_cast を追加
  cout << c << endl;

  // 基底クラス(Base)のポインタから派生クラス(Sub)のポインタに変換
  Base* base = new Sub;
  Sub* sub = static_cast<Sub*>(base); // ここに static_cast を追加(ここだけ dynamic_cast でもよい)
  cout << sub->b << ' ' << sub->s << endl;
  delete base;
}


### ❓️問題１１ 人間と幽霊と木

ダイナさんは、ゲームに登場するオブジェクトをあらわすクラスを使うプログラムを作っています。オブジェクトは以下の3つのクラスから選びます。

1. `Human`(ヒューマン)
2. `Ghost`(ゴースト)
3. `Tree`(ツリー)

また、各オブジェクト・クラスは、以下の機能を持つクラスを必要なだけ継承しています。

* `Movable`(ムーバブル): 移動できる
* `Talkable`(トーカブル): 会話できる
* `Collidable`(コライダブル): 衝突できる

例えば`Ghost`クラスは移動と会話ができますが、衝突はできません。そこで、`Movable`と`Talkable`の2つだけを継承しています。

現在のプログラムでは、`main`関数でオブジェクトの種類を入力させて、種類に対応したオブジェクトを作る、というところまで作れています。

あとは、作成したオブジェクトについて、`dynamic_cast`を使って継承しているクラスを調べ、継承している場合はその機能クラスの仮想関数を呼び出したら完成です。

ですが、ダイナさんはここで力尽きてしまいました。<br>
ダイナさんの代わりに、`dynamic_cast`を使うプログラムを完成させてください。

<br><details><summary>🗝️(クリックでヒントを見る)</summary>

**プログラム例**

1. `Movable*`型の変数`m`を宣言し、`dynamic_cast`で変数`p`を`Movable*`型に変換して代入する
2. if文を使って、変数`m`が`nullptr`でなければ、`m`の仮想関数`Move`を呼び出す
3. `Talkable*`型の変数`t`を宣言し、`dynamic_cast`で変数`p`を`Talkable*`型に変換して代入する
4. if文を使って、変数`t`が`nullptr`でなければ、`t`の仮想関数`Talk`を呼び出す
5. `Collidable*`型の変数`c`を宣言し、`dynamic_cast`で変数`p`を`Collidable*`型に変換して代入する
6. if文を使って、変数`c`が`nullptr`でなければ、`c`の仮想関数`Collide`を呼び出す

</details><br>

**入力データ例（１）**

```txt
2
```

**出力例（１）**

```txt
幽霊は移動できる・・・多分。
「うらめしやー」
```

**入力データ例（２）**

```txt
1
```

**出力例（２）**

```txt
人間は移動できる。
「こんにちは」
人にぶつかった。「ごめんなさい」
```

**入力データ例（３）**

```txt
2
```

**出力例（３）**

```txt
木にぶつかった。
```


In [ ]:
%%writefile practice_04c.cpp
#include <iostream>
using namespace std;

#include <iostream>
using namespace std;

// 移動機能を付けるクラス
struct Movable { virtual void Move() = 0; };

// 会話機能を付けるクラス
struct Talkable { virtual void Talk() = 0; };

// 衝突判定を付けるクラス
struct Collidable { virtual void Collide() = 0; };

// 基本クラス
struct Object { virtual ~Object() = default; };

// 人間クラス(可能な操作: 移動 会話 衝突)
struct Human : Object, Movable, Talkable, Collidable {
  void Move() override { cout << "人間は移動できる。" << endl; }
  void Talk() override { cout << "「こんにちは」" << endl; }
  void Collide() override { cout << "人にぶつかった。「ごめんなさい」" << endl; }
};

// 幽霊クラス(可能な操作: 移動 会話)
struct Ghost : Object, Movable, Talkable {
  void Move() override { cout << "幽霊は移動できる・・・多分。" << endl; }
  void Talk() override { cout << "「うらめしやー」" << endl; }
};

// 木クラス(可能な操作: 衝突)
struct Tree : Object, Collidable {
  void Collide() override { cout << "木にぶつかった。" << endl; }
};

int main() {

  Object* p = nullptr;

  int a;
  cin >> a;
  switch (a) {
    case 1: p = new Human; break;
    case 2: p = new Ghost; break;
    case 3: p = new Tree; break;
  }
  // ここから上は変更しない

  // この下に、3つの機能クラスを継承しているかどうかを調べて、仮想関数を呼び出すプログラムを書く


  // ここから下は変更しない
  delete p;
}


In [ ]:
# @title 動作テスト
!cat <(echo) practice_04c.cpp > practice_04c_.cpp
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_04c practice_04c_.cpp && echo "この下をクリックして、オブジェクトの種類1, 2, 3のどれかを入力" && ./practice_04c
!rm practice_04c_.cpp

In [ ]:
# @title 実行
!diff -Z <(echo -e "人間は移動できる。\n「こんにちは」\n人にぶつかった。「ごめんなさい」\n幽霊は移動できる・・・多分。\n「うらめしやー」\n木にぶつかった。") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_04c practice_04c.cpp && echo "1" | ./practice_04c && echo "2" | ./practice_04c && echo "3" | ./practice_04c) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_04c.cpp
#include <iostream>
using namespace std;

// 移動機能を付けるクラス
struct Movable { virtual void Move() = 0; };

// 会話機能を付けるクラス
struct Talkable { virtual void Talk() = 0; };

// 衝突判定を付けるクラス
struct Collidable { virtual void Collide() = 0; };

// 基本クラス
struct Object { virtual ~Object() = default; };

// 人間クラス(可能な操作: 移動 会話 衝突)
struct Human : Object, Movable, Talkable, Collidable {
  void Move() override { cout << "人間は移動できる。" << endl; }
  void Talk() override { cout << "「こんにちは」" << endl; }
  void Collide() override { cout << "人にぶつかった。「ごめんなさい」" << endl; }
};

// 幽霊クラス(可能な操作: 移動 会話)
struct Ghost : Object, Movable, Talkable {
  void Move() override { cout << "幽霊は移動できる・・・多分。" << endl; }
  void Talk() override { cout << "「うらめしやー」" << endl; }
};

// 木クラス(可能な操作: 衝突)
struct Tree : Object, Collidable {
  void Collide() override { cout << "木にぶつかった。" << endl; }
};

int main() {

  Object* p = nullptr;

  int a;
  cin >> a;
  switch (a) {
    case 1: p = new Human; break;
    case 2: p = new Ghost; break;
    case 3: p = new Tree; break;
  }
  // ここから上は変更しない

  // この下に、3つの機能クラスを継承しているかどうかを調べて、仮想関数を呼び出すプログラムを書く

  Movable* m = dynamic_cast<Movable*>(p);
  if (m) {
    m->Move();
  }

  Talkable* t = dynamic_cast<Talkable*>(p);
  if (t) {
    t->Talk();
  }

  Collidable* c = dynamic_cast<Collidable*>(p);
  if (c) {
    c->Collide();
  }

  // ここから下は変更しない
  delete p;
}


### ❓️問題１２ 数値か、文章か

「数値データ」か「文章データ」のどちらかを返す`InputData`関数があります。

数値データは`NumberData`(ナンバー・データ)型、文章データは`TextData`(テキスト・データ)型であらわされます。どちらの型も、先頭のメンバ変数は`Header`(ヘッダ)型です。

`InputData`関数の戻り値は`Header*`型ですが、`type`メンバ変数を見ることで、実際の型を判別できます。

以下のプログラムに、戻り値が数値データの場合は「`number:数値メンバの値`」、文章データの場合は「`text:文章メンバの値`」を出力するプログラムを追加してください。

<br><details><summary>🗝️(クリックでヒントを見る)</summary>

**プログラム例**

1. if文を使って、`header->type`が`DataType::number`と等しい場合、以下の処理を行う
   1. `NumberData*`型のポインタ変数`p`を宣言する
   2. `reinterpret_cast`を使って、`header`を`NumberData*`にキャストし、`p`に代入する
   3. `cout`に文字列`"number:"`、変数`p->number`を出力し、改行する
   4. `p`を`delete`する
2. else句を使って、以下の処理を行う
   1. `TextData*`型のポインタ変数`p`を宣言する
   2. `reinterpret_cast`を使って、`header`を`TextData*`にキャストし、`p`に代入する
   3. `cout`に文字列`"text:"`、変数`p->text`を出力し、改行する
   4. `p`を`delete`する

</details><br>

**入力データ例（１）**

```txt
cast
```

**出力例（１）**

```txt
text:cast
```

**入力データ例（２）**

```txt
365
```

**出力例（２）**

```txt
number:365
```


In [ ]:
%%writefile practice_04d.cpp
#include <iostream>
using namespace std;

// データの種類
enum class DataType {
  number,
  text,
};

// データの共通情報クラス
struct Header {
  DataType type; // データの種類
};

// 数値データクラス
struct NumberData {
  Header header; // 共通情報
  int number;    // 数値
};

// 文章データクラス
struct TextData {
  Header header; // 共通情報
  char text[32]; // 文章
};

// NumberData または TextData を返す関数(定義は別ファイル)
// 返された型は、戻り値の Header::type メンバ変数で判断する
//   - DataType::number の場合: 戻り値を NumberData 型にキャストする
//   - DataType::text の場合:   戻り値を TextData 型にキャストする
//
// ※戻り値を使い終わったら、キャストした型で delete すること
Header* InputData();

int main() {

  Header* header = InputData();
  // ここから上は変更しない

  // この下に、データの種類によって数値か文章を出力するプログラムを書く


}


In [ ]:
# @title 動作テスト
!echo -e "#include <stdio.h> \n #include <stdlib.h> \n enum class DataType { number, text, }; struct Header { DataType type; }; struct NumberData { Header header; int number; }; struct TextData { Header header; char text[32]; }; Header* InputData() { char s[32]; if (scanf(\"%31s\", s) != 1) { s[0] = 0; } ; if (s[0] >= '1' && s[0] <= '9') { NumberData* p = new NumberData; p->header.type = DataType::number; p->number = atoi(s); return &p->header; } else { TextData* p = new TextData; p->header.type = DataType::text; char* a = p->text; const char* b = s; while (*b) { *a = *b; a++; b++; } *a = 0; return &p->header; } }" > data.cpp
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_04d data.cpp practice_04d.cpp && ./practice_04d
!rm data.cpp

In [ ]:
# @title 実行
!echo -e "#include <stdio.h> \n #include <stdlib.h> \n enum class DataType { number, text, }; struct Header { DataType type; }; struct NumberData { Header header; int number; }; struct TextData { Header header; char text[32]; }; Header* InputData() { char s[32]; if (scanf(\"%31s\", s) != 1) { s[0] = 0; } ; if (s[0] >= '1' && s[0] <= '9') { NumberData* p = new NumberData; p->header.type = DataType::number; p->number = atoi(s); return &p->header; } else { TextData* p = new TextData; p->header.type = DataType::text; char* a = p->text; const char* b = s; while (*b) { *a = *b; a++; b++; } *a = 0; return &p->header; } }" > data.cpp
!diff -Z <(echo -e "number:98765432\ntext:reinterpret_cast") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_04d practice_04d.cpp data.cpp && echo "98765432" | ./practice_04d && echo "reinterpret_cast" | ./practice_04d) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"
!rm data.cpp

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_04d.cpp
#include <iostream>
using namespace std;

// データの種類
enum class DataType {
  number,
  text,
};

// データの共通情報クラス
struct Header {
  DataType type; // データの種類
};

// 数値データクラス
struct NumberData {
  Header header; // 共通情報
  int number;    // 数値
};

// 文章データクラス
struct TextData {
  Header header; // 共通情報
  char text[32]; // 文章
};

// NumberData または TextData を返す関数(定義は別ファイル)
// 返された型は、戻り値の Header::type メンバ変数で判断する
//   - DataType::number の場合: 戻り値を NumberData 型にキャストする
//   - DataType::text の場合:   戻り値を TextData 型にキャストする
Header* InputData();

int main() {

  Header* header = InputData();
  // ここから上は変更しない

  // この下に、データの種類によって数値か文章を出力するプログラムを書く

  if (header->type == DataType::number) {
    NumberData* p = reinterpret_cast<NumberData*>(header);
    cout << "number:" << p->number << endl;
    delete p;
  }
  else {
    TextData* p = reinterpret_cast<TextData*>(header);
    cout << "text:" << p->text << endl;
    delete p;
  }
}


### ❓️問題１３ マクロ・オンリー

マクロを使ってC++言語に見えないプログラムを書こうとしています。<br>
プログラムに書かれている5種類のマクロだけを使って、以下の処理を行うプログラムを完成させてください。

**処理内容**

1. 標準入力からデータ数`n`を読み込む
2. `n`個の`int`型配列を宣言する
3. 配列にデータを読み込む
4. 配列の内容を出力する
5. 配列の内容を逆順で出力する

**マクロ一覧**

* `IN`(イン): 標準入力からデータ数を読み込む
* `VEC`(ベク): 配列を宣言する
* `IN_VEC`(イン・ベク): 標準入力から配列にデータを読み込む
* `PRINT`(プリント): 配列の内容を標準出力に出力する
* `PRINT_REVERSE`(プリント・リバース): 配列の内容を標準出力に逆順で出力する

**入力データ例**

```txt
5
1 2 3 4 5
```

**出力例**

```txt
1 2 3 4 5
5 4 3 2 1
```


In [ ]:
%%writefile practice_05a.cpp
#include <iostream>
#include <vector>
using namespace std;

// int 型の変数 var を宣言し、 cin から var にデータを読み込む
#define IN(var) int var; cin >> var;

// int 型の配列 var を、要素数 count で宣言する
#define VEC(var, count) vector<int> var(count);

// cin から配列 var にデータを読み込む
#define IN_VEC(var) for (int i = 0; i < (int)var.size(); i++) { cin >> var[i]; }

// 配列 var の内容を cout に出力する
#define PRINT(var) \
  for (auto i = var.begin(); i != var.end(); i++) { \
    cout << *i << ' '; \
  } cout << endl;

// 配列 var の内容を逆順で cout に出力する
#define PRINT_REVERSE(var) \
  for (auto i = var.rbegin(); i != var.rend(); i++) { \
    cout << *i << ' '; \
  } cout << endl;

int main() {
  // ここから上は変更しない

  // この下に、マクロだけを使って配列を読み書きするプログラムを書く

}


In [ ]:
# @title 動作テスト
!g++ -std=c++20 -O2 -Wall -Wextra -o practice_05a practice_05a.cpp && echo "この下をクリックして、データ数とデータリストを入力" && ./practice_05a

In [ ]:
# @title 実行
!diff -Z <(echo -e "1 2 3 4 5\n5 4 3 2 1\n57\n57\n3 1 4 1 5 9 2 6 5 3 5\n5 3 5 6 2 9 5 1 4 1 3") <(g++ -std=c++20 -O2 -Wall -Wextra -o practice_05a practice_05a.cpp && echo "5 1 2 3 4 5" | ./practice_05a && echo "1 57" | ./practice_05a && echo "11 3 1 4 1 5 9 2 6 5 3 5" | ./practice_05a) > /dev/null && test $? -eq 0 && echo -e "\033[32;1mAC" || echo -e "\033[31;1mWA"

In [ ]:
# @title 🔒解答例(どうしても問題を解けない場合に見てください)
%%writefile practice_05a.cpp
#include <iostream>
#include <vector>
using namespace std;

// int 型の変数 var を宣言し、 cin から var にデータを読み込む
#define IN(var) int var; cin >> var;

// int 型の配列 var を、要素数 count で宣言する
#define VEC(var, count) vector<int> var(count);

// cin から配列 var にデータを読み込む
#define IN_VEC(var) for (int i = 0; i < (int)var.size(); i++) { cin >> var[i]; }

// 配列 var の内容を cout に出力する
#define PRINT(var) \
  for (auto i = var.begin(); i != var.end(); i++) { \
    cout << *i << ' '; \
  } cout << endl;

// 配列 var の内容を逆順で cout に出力する
#define PRINT_REVERSE(var) \
  for (auto i = var.rbegin(); i != var.rend(); i++) { \
    cout << *i << ' '; \
  } cout << endl;

int main() {
  // ここから上は変更しない

  // この下に、マクロだけを使って配列を読み書きするプログラムを書く
  IN(n)
  VEC(v, n)
  IN_VEC(v)
  PRINT(v)
  PRINT_REVERSE(v)
}
